# AccessPoint — Full Pipeline : DIFS → CBAG → SBAR

Same structure as the UAV1 pipeline, using **AccessPoint's own working CBAG configuration** (gates, EVAL_CAP, generators unchanged).

**SECTION 1** DIFS+CBAG (phase-cache p0→p1→p2) · **SECTION 2** SBAR held-out (pause/resume) · **SECTION 3** all charts → PDF+TikZ · **SECTION 4** zip.

Notes: AccessPoint uses EVAL_CAP=400, Gate-1 0.5, Gate-3 operating 0.10 (SBAR trains at L1<0.25).

# SECTION 1 — DIFS + CBAG (AccessPoint configuration, unchanged)

### 1.1 — CONFIG + imports + cache helpers

In [ ]:
# ===== CELL 1 — CONFIG + imports + cache helpers =====
import os,glob,gc,json,pickle,warnings,time,numpy as np,pandas as pd
warnings.filterwarnings("ignore")
try:
    import torch, torch.nn as nn; HAVE_TORCH=True; torch.set_num_threads(os.cpu_count() or 4)
except Exception:
    HAVE_TORCH=False
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from scipy.stats import wilcoxon
def P(*a): print(*a,flush=True)
SEED=42; np.random.seed(SEED)
if HAVE_TORCH: torch.manual_seed(SEED)

# ---------------- CONTROL PANEL ----------------
DATASET   = "AccessPoint"          # "UAV1" | "AccessPoint" | "UAV3_GSC"  <-- confirmed working, real Normal class 20.9%
SUBSAMPLE = False          # FULL data (chunked loader below keeps RAM low)
PER_CLASS_SUB = 30000
TARGET    = "cnn"           # "cnn" (CNN-1D, needs torch) else auto MLP fallback
CNN_PER_CLASS = 5500; CNN_EPOCHS = 6
GENERATORS = ["C-SA","C-RL","C-PSO","C-GA","C-BO","CG-Diffusion","CC-CGAN","HopSkipJump","Sign-OPT"]  # FULL proposal roster
# generator TYPE for the CBAG paper: constrained (ours) vs unconstrained baselines
GEN_CLASS={"C-SA":"constrained","C-RL":"constrained","C-PSO":"constrained","C-GA":"constrained",
           "C-BO":"constrained","CG-Diffusion":"constrained","CC-CGAN":"constrained",
           "HopSkipJump":"baseline","Sign-OPT":"baseline"}

ENABLE_HEAVY = True    # run heavy methods (C-BO/CG-Diffusion/CC-CGAN/HopSkipJump/Sign-OPT); False -> skip (BO/Diff use projected)
EVAL_CAP  = 4000             # detected samples/class fed to each generator
# ---- Tri-Gate (targeted) ----
# EXTRACT survivors at ANY gate config you want (no regeneration):
# phase3(theta_mode="fixed", theta_fix=0.5, gate2_cmr=False, gate3=0.5, save_name="survivors_L1_0.5")

GATE1_THR = 0.5            # Gate-1 evasion: P(Normal) >= this
GATE2_CMR = False           # Gate-2 = oracle only (True = oracle & CMR, func2 original)
THETA_MODE= "fixed"         # "fixed" -> THETA_FIX ; "paper" -> max(0.5, p10)
THETA_FIX = 0.50
GATE3_OP  = 0.10           # Gate-3 operating point (stealthy)
G3_SWEEP  = [0.10,0.25,0.50,1.00,2.00]
HARD_SD_K = 3.0; CMR_PCT = 90
IMMUTABLE={'ip.proto','ip.ttl','ip.dst','id.orig_h','id.resp_h'}
UNSETTABLE={'radiotap.dbm_antsignal','wlan_radio.signal_dbm','radiotap.datarate',
 'radiotap.channel.freq','wlan_radio.frequency','radiotap.channel.flags.ofdm'}
LEAKAGE=['frame.number','frame.time_relative','frame.time_epoch','radiotap.timestamp.ts','radiotap.mactime',
 'wlan_radio.start_tsf','wlan_radio.end_tsf','wlan.seq','radiotap.length','uid','id.resp_h','id.orig_p','id.resp_p']
DATASETS={
 "UAV1":       dict(glob="/kaggle/input/datasets/saleemsarg/cbag-uav/UAV-Case1-Label (1).csv", label="Label", benign="Normal",
    drop=["Reconnassiance","Reconnaissance"],
    IG=['radiotap.channel.flags.cck','frame.encap_type','radiotap.channel.freq','ip.proto','ip.ttl','wlan.fixed.capabilities.ess','ip.dst','frame.len','udp.dstport','wlan.fc.subtype'],
    LIME=['radiotap.channel.flags.cck','wlan.fixed.beacon','arp','radiotap.channel.flags.ofdm','wlan.fc.subtype','tcp.ack','wlan.fc.retry','frame.time_delta_displayed','wlan.bssid','udp.length'],
    craft=['wlan.fc.retry','wlan.fcs.bad_checksum','wlan.rsn.capabilities.mfpc','frame.encap_type']),
 "AccessPoint":dict(glob="/kaggle/input/datasets/saleemsarg/cbag-uav/Access Point Case2 Label.csv", label="Normal", benign="Normal",
    drop=["Reconnaissance","Ewil Twin"],
    IG=['radiotap.rxflags','wlan.fcs.bad_checksum','ip.ttl','ip.proto','udp.dstport','frame.len','radiotap.channel.flags.cck','wlan.fc.subtype','wlan.fc.retry','udp.length'],
    LIME=['radiotap.rxflags','wlan.fc.pwrmgt','wlan.fc.retry','udp.length','nbss.continuation_data','wlan.fc.subtype','radiotap.channel.flags.cck','nbns','radiotap.present.tsft','wlan_radio.data_rate'],
    craft=['wlan.fc.retry','wlan.fcs.bad_checksum','wlan.fc.pwrmgt','frame.encap_type']),
 "UAV3_GSC":   dict(glob="/kaggle/input/datasets/saleemsarg/cbag-uav/GSC Case3 Label .csv", label="Class", benign="Benign",
    drop=["Evil Twin"],
    IG=['fwd_header_size_min','idle.std','bwd_header_size_tot','bwd_header_size_min','fwd_header_size_tot','bwd_header_size_max','bwd_pkts_per_sec','bwd_init_window_size','bwd_data_pkts_tot','id.orig_h'],
    LIME=['fwd_header_size_min','fwd_PSH_flag_count','fwd_pkts_payload.avg','fwd_pkts_payload.max','bwd_header_size_tot','flow_ECE_flag_count','bwd_pkts_payload.avg','bwd_pkts_payload.tot','fwd_pkts_payload.min','bwd_pkts_payload.std'],
    craft=[]),
}
cfg=DATASETS[DATASET]; BEN=cfg["benign"]; craft=cfg["craft"]
CACHE=f"cbag_cache_{DATASET}"; OUT=f"cbag_out_{DATASET}"; os.makedirs(CACHE,exist_ok=True); os.makedirs(OUT,exist_ok=True)
def feature_pool(feat,difs): return [f for f in difs if f not in UNSETTABLE]
def has(t): return os.path.exists(f"{CACHE}/{t}.pkl")
def save(t,o): pickle.dump(o,open(f"{CACHE}/{t}.pkl","wb"))
def load(t): return pickle.load(open(f"{CACHE}/{t}.pkl","rb"))
P(f"CONFIG ok. DATASET={DATASET} SUBSAMPLE={SUBSAMPLE} TARGET={TARGET} torch={HAVE_TORCH} GENERATORS={GENERATORS}")


### 1.3 — generators (full roster)

In [ ]:
# ===== CELL 2 — REAL generators (inlined; no external file) =====
def C_SA(det,lo,hi,mut,pN,L1,rng,wp=0.3,iters=400,cool=0.98,step=0.5,track=False):
    Xa=det.copy()
    for j in mut: Xa[:,j]=np.clip(Xa[:,j],lo[j],hi[j])
    en=lambda Z:-pN(Z)+wp*L1(Z,det,mut); E=en(Xa); T=1.0; curve=[]
    for it in range(iters):
        c=Xa.copy(); jj=np.array(mut)[rng.integers(0,len(mut),len(Xa))]
        c[np.arange(len(Xa)),jj]+=rng.normal(0,step,len(Xa)).astype(np.float32)
        for j in mut: c[:,j]=np.clip(c[:,j],lo[j],hi[j])
        Ec=en(c); dE=Ec-E; acc=(dE<0)|(rng.random(len(Xa))<np.exp(-np.clip(dE,0,50)/max(T,1e-3)))
        Xa[acc]=c[acc]; E[acc]=Ec[acc]; T*=cool
        if track and (it%20==0 or it==iters-1): curve.append([it,float((pN(Xa)>=GATE1_THR).mean()*100)])
    return (Xa,iters,curve) if track else (Xa,iters)

def C_PSO(det,lo,hi,mut,pN,L1,rng,wp=0.3,Pn=12,iters=40):
    N,d,F=len(det),len(mut),det.shape[1]; base=np.repeat(det[:,None,:],Pn,1).astype(np.float32)
    base[:,:,mut]=rng.uniform(lo[mut],hi[mut],(N,Pn,d)).astype(np.float32); vel=np.zeros((N,Pn,d),np.float32)
    en=lambda B:-pN(B.reshape(-1,F)).reshape(N,Pn)+wp*(np.abs(B[:,:,mut]-det[:,None,mut])/(hi[mut]-lo[mut]+1e-9)).sum(2)
    E=en(base); pb=base.copy(); pbE=E.copy(); gi=pbE.argmin(1); gb=pb[np.arange(N),gi].copy()
    for _ in range(iters):
        r1=rng.random((N,Pn,d)).astype(np.float32); r2=rng.random((N,Pn,d)).astype(np.float32)
        vel=0.6*vel+1.5*r1*(pb[:,:,mut]-base[:,:,mut])+1.5*r2*(gb[:,None,mut]-base[:,:,mut])
        base[:,:,mut]=np.clip(base[:,:,mut]+vel,lo[mut],hi[mut]); E=en(base)
        imp=E<pbE; pb[imp]=base[imp]; pbE[imp]=E[imp]; gi=pbE.argmin(1); gb=pb[np.arange(N),gi].copy()
    return gb, Pn*iters

def C_GA(det,lo,hi,mut,pN,L1,rng,wp=0.3,Pn=12,gens=40):
    N,d,F=len(det),len(mut),det.shape[1]; pop=np.repeat(det[:,None,:],Pn,1).astype(np.float32)
    pop[:,:,mut]=rng.uniform(lo[mut],hi[mut],(N,Pn,d)).astype(np.float32)
    fit=lambda B:-(-pN(B.reshape(-1,F)).reshape(N,Pn)+wp*(np.abs(B[:,:,mut]-det[:,None,mut])/(hi[mut]-lo[mut]+1e-9)).sum(2))
    for _ in range(gens):
        f=fit(pop); order=np.argsort(-f,1); elite=np.take_along_axis(pop,order[:,:Pn//2,None],1); off=elite.copy()
        msk=rng.random((N,Pn//2,d))<0.3
        off[:,:,mut]=np.clip(off[:,:,mut]+msk*rng.normal(0,0.3,(N,Pn//2,d)).astype(np.float32),lo[mut],hi[mut])
        pop=np.concatenate([elite,off],1)
    f=fit(pop); gi=f.argmax(1); return pop[np.arange(N),gi], Pn*gens

def C_RL(det,tr,lo,hi,mut,pN,L1,rng,wp=0.3,T=6,iters=40,step=0.35):
    if not HAVE_TORCH: return C_SA(det,lo,hi,mut,pN,L1,rng,wp=wp)   # SA fallback if no torch
    d=len(mut)
    class Pol(nn.Module):
        def __init__(s): super().__init__(); s.n=nn.Sequential(nn.Linear(d,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh()); s.mu=nn.Linear(64,d); s.ls=nn.Parameter(torch.full((d,),-0.5))
        def forward(s,x): h=s.n(x); return s.mu(h),s.ls.exp()
    pol=Pol(); opt=torch.optim.Adam(pol.parameters(),3e-3); loj,hij=lo[mut],hi[mut]
    for _ in range(iters):
        b=tr[rng.choice(len(tr),min(256,len(tr)),replace=False)].copy(); x=b.copy(); lps=[]
        for _ in range(T):
            mu,sd=pol(torch.tensor(x[:,mut])); dist=torch.distributions.Normal(mu,sd); a=dist.sample()
            lps.append(dist.log_prob(a).sum(1)); x[:,mut]=np.clip(x[:,mut]+step*a.numpy(),loj,hij)
        R=torch.tensor(pN(x)-wp*L1(x,b,mut),dtype=torch.float32); adv=R-R.mean()
        loss=-(torch.stack(lps).sum(0)*adv).mean(); opt.zero_grad(); loss.backward(); opt.step()
    x=det.copy()
    for _ in range(T):
        with torch.no_grad(): mu,sd=pol(torch.tensor(x[:,mut]))
        x[:,mut]=np.clip(x[:,mut]+step*mu.numpy(),loj,hij)
    return x, iters*T

HSJ_MAX=60; BO_MAX=60   # per-class sample cap for the slow per-sample methods (raise for more samples on GPU)
def C_BO(det,lo,hi,mut,pN,L1,rng,wp=0.3,init=8,iters=20):
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import Matern
    from scipy.stats import norm as _norm
    d=len(mut); loj=lo[mut]; hij=hi[mut]; out=det.copy()
    for idx in range(min(len(det),BO_MAX)):
        x0=det[idx]
        def obj(delta):
            c=x0.copy(); c[mut]=np.clip(delta,loj,hij); z=c[None,:]; return float(-pN(z)[0]+wp*L1(z,x0[None,:],mut)[0])
        Xs=rng.uniform(loj,hij,(init,d)); Ys=np.array([obj(v) for v in Xs])
        for _ in range(iters):
            gp=GaussianProcessRegressor(kernel=Matern(nu=2.5),normalize_y=True,alpha=1e-4,n_restarts_optimizer=0).fit(Xs,Ys)
            cand=rng.uniform(loj,hij,(256,d)); mu,sd=gp.predict(cand,return_std=True)
            imp=Ys.min()-mu; Z=imp/(sd+1e-9); ei=imp*_norm.cdf(Z)+sd*_norm.pdf(Z)
            nx=cand[ei.argmax()]; Xs=np.vstack([Xs,nx]); Ys=np.append(Ys,obj(nx))
        c=x0.copy(); c[mut]=np.clip(Xs[Ys.argmin()],loj,hij); out[idx]=c
    return out, init+iters

def CG_Diffusion(det,tr,lo,hi,mut,pN,L1,rng,wp=0.3,steps=16,epochs=40,guide=15):
    if not HAVE_TORCH: return C_SA(det,lo,hi,mut,pN,L1,rng,wp=wp)
    d=len(mut); w=(hi[mut]-lo[mut]+1e-9); nrm=lambda a:(a-lo[mut])/w; dn=lambda a:a*w+lo[mut]
    Tr=torch.tensor(np.clip(nrm(tr[:,mut]),0,1).astype(np.float32))
    betas=torch.linspace(1e-4,0.05,steps); ab=torch.cumprod(1-betas,0)
    net=nn.Sequential(nn.Linear(d+1,128),nn.SiLU(),nn.Linear(128,128),nn.SiLU(),nn.Linear(128,d)); opt=torch.optim.Adam(net.parameters(),2e-3)
    for _ in range(epochs):
        i=torch.randint(0,len(Tr),(min(256,len(Tr)),)); x0=Tr[i]; t=torch.randint(0,steps,(len(x0),)); a=ab[t][:,None]
        nz=torch.randn_like(x0); xt=torch.sqrt(a)*x0+torch.sqrt(1-a)*nz
        loss=((net(torch.cat([xt,t[:,None].float()/steps],1))-nz)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
    x=torch.tensor(np.clip(nrm(det[:,mut]),0,1).astype(np.float32)); x=torch.sqrt(ab[-1])*x+torch.sqrt(1-ab[-1])*torch.randn_like(x)
    with torch.no_grad():
        for t in reversed(range(steps)):
            tt=torch.full((len(x),),t); eps=net(torch.cat([x,tt[:,None].float()/steps],1)); a=ab[t]
            x0h=torch.clamp((x-torch.sqrt(1-a)*eps)/torch.sqrt(a),0,1)
            x=x0h if t==0 else torch.sqrt(ab[t-1])*x0h+torch.sqrt(1-ab[t-1])*torch.randn_like(x)
    cur=det.copy(); cur[:,mut]=np.clip(dn(x.numpy()),lo[mut],hi[mut]); E=-pN(cur)+wp*L1(cur,det,mut)
    for _ in range(guide):
        c=cur.copy(); jj=np.array(mut)[rng.integers(0,len(mut),len(cur))]; c[np.arange(len(cur)),jj]+=rng.normal(0,0.4,len(cur)).astype(np.float32)
        for j in mut: c[:,j]=np.clip(c[:,j],lo[j],hi[j])
        Ec=-pN(c)+wp*L1(c,det,mut); acc=Ec<E; cur[acc]=c[acc]; E[acc]=Ec[acc]
    return cur, steps*epochs+guide

def CC_CGAN(det,tr,lo,hi,mut,pN,L1,rng,wp=0.3,epochs=60,zdim=8,guide=15):
    if not HAVE_TORCH: return C_GA(det,lo,hi,mut,pN,L1,rng,wp=wp)
    d=len(mut); w=(hi[mut]-lo[mut]+1e-9); nrm=lambda a:(a-lo[mut])/w; dn=lambda a:a*w+lo[mut]
    R=torch.tensor(np.clip(nrm(tr[:,mut]),0,1).astype(np.float32))
    G=nn.Sequential(nn.Linear(zdim,64),nn.ReLU(),nn.Linear(64,d),nn.Sigmoid()); D=nn.Sequential(nn.Linear(d,64),nn.LeakyReLU(0.2),nn.Linear(64,1))
    og=torch.optim.Adam(G.parameters(),2e-3); od=torch.optim.Adam(D.parameters(),2e-3); bce=nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        idx=torch.randint(0,len(R),(min(256,len(R)),)); real=R[idx]; z=torch.randn(len(real),zdim); fake=G(z).detach()
        od.zero_grad(); (bce(D(real),torch.ones(len(real),1))+bce(D(fake),torch.zeros(len(real),1))).backward(); od.step()
        og.zero_grad(); z=torch.randn(len(real),zdim); bce(D(G(z)),torch.ones(len(real),1)).backward(); og.step()
    with torch.no_grad(): gen=G(torch.randn(len(det),zdim)).numpy()
    cur=det.copy(); cur[:,mut]=np.clip(dn(gen),lo[mut],hi[mut]); E=-pN(cur)+wp*L1(cur,det,mut)
    for _ in range(guide):
        c=cur.copy(); jj=np.array(mut)[rng.integers(0,len(mut),len(cur))]; c[np.arange(len(cur)),jj]+=rng.normal(0,0.4,len(cur)).astype(np.float32)
        for j in mut: c[:,j]=np.clip(c[:,j],lo[j],hi[j])
        Ec=-pN(c)+wp*L1(c,det,mut); acc=Ec<E; cur[acc]=c[acc]; E[acc]=Ec[acc]
    return cur, epochs+guide

def _rand_unc(det,lo,hi,mut,pN,L1,rng,**k):
    x=det.copy(); x[:,mut]=rng.uniform(lo[mut],hi[mut],(len(det),len(mut))).astype(np.float32); return x,1

def HopSkipJump(det,lo,hi,mut,pN,L1,rng,ref=None,iters=12,B=18,**k):
    N=min(len(det),HSJ_MAX); out=det.copy(); ref=det.mean(0) if ref is None else ref
    for i in range(N):
        orig=det[i]; adv=ref.copy()
        if pN(adv[None])[0]<0.5: out[i]=orig; continue
        for _ in range(iters):
            a,b=0.0,1.0
            for _ in range(8):
                m=(a+b)/2; c=orig.copy(); c[mut]=orig[mut]+m*(adv[mut]-orig[mut]); (a,b)=((a,m) if pN(c[None])[0]>=0.5 else (m,b))
            bnd=orig.copy(); bnd[mut]=orig[mut]+b*(adv[mut]-orig[mut])
            U=rng.normal(0,1,(B,len(mut))); U/=np.linalg.norm(U,axis=1,keepdims=True)+1e-9
            cand=np.repeat(bnd[None],B,0).astype(np.float32); cand[:,mut]=np.clip(cand[:,mut]+0.05*U,lo[mut],hi[mut])
            phi=(pN(cand)>=0.5).astype(float)*2-1; grad=(phi[:,None]*U).mean(0); adv=bnd.copy(); adv[mut]=np.clip(bnd[mut]+0.1*grad,lo[mut],hi[mut])
        out[i]=adv
    return out, iters*(8+B)

def Sign_OPT(det,lo,hi,mut,pN,L1,rng,ref=None,iters=12,Q=20,**k):
    N=min(len(det),HSJ_MAX); out=det.copy(); ref=det.mean(0) if ref is None else ref
    for i in range(N):
        orig=det[i]; th=(ref[mut]-orig[mut]); th/=(np.linalg.norm(th)+1e-9)
        def g(t):
            a,b=0.0,4.0
            for _ in range(10):
                m=(a+b)/2; c=orig.copy(); c[mut]=np.clip(orig[mut]+m*t,lo[mut],hi[mut]); (a,b)=((a,m) if pN(c[None])[0]>=0.5 else (m,b))
            return b
        g0=g(th)
        for _ in range(iters):
            u=rng.normal(0,1,len(mut)); u/=np.linalg.norm(u)+1e-9; s=1 if g(th+0.1*u)<g0 else -1
            th=th+0.05*s*u; th/=np.linalg.norm(th)+1e-9; g0=g(th)
        c=orig.copy(); c[mut]=np.clip(orig[mut]+g0*th,lo[mut],hi[mut]); out[i]=c
    return out, iters*Q

GENS={"C-SA":C_SA,"C-RL":C_RL,"C-PSO":C_PSO,"C-GA":C_GA,"C-BO":C_BO,"CG-Diffusion":CG_Diffusion,"CC-CGAN":CC_CGAN,"HopSkipJump":HopSkipJump,"Sign-OPT":Sign_OPT}
NEEDS_TR={"C-RL","CG-Diffusion","CC-CGAN"}
HEAVY={"C-BO","CG-Diffusion","CC-CGAN","HopSkipJump","Sign-OPT"}
P("generators inlined (full proposal roster):",list(GENS))


### 1.4 — PHASE 0 — load & clean

In [ ]:
# ===== CELL 3 — PHASE 0: load & clean (CHUNKED / low-RAM, FULL data, no sampling) =====
def _hexnum(s):
    num=pd.to_numeric(s,errors="coerce"); m=num.isna()&s.notna()
    if m.any(): num.loc[m]=s[m].map(lambda v: float(int(str(v).strip(),16)) if str(v).strip().lower().startswith("0x") else np.nan)
    return num
def phase0(force=False):
    if has("p0") and not force: P("[P0] cached"); return load("p0")
    files=sorted(glob.glob(cfg["glob"])); assert files,"NO FILES at "+cfg["glob"]
    # PASS 1 (sample only): detect string/categorical columns for consistent factorization
    samp=pd.read_csv(files[0],low_memory=False,nrows=200000).rename(columns={cfg["label"]:"Label"}).drop(columns=["Label"])
    cols=list(samp.columns)
    catcols=set(c for c in cols if (_hexnum(samp[c]).isna()&samp[c].notna()).any())
    catmap={c:{} for c in catcols}; del samp; gc.collect()
    # PASS 2 (chunked): convert each chunk to float32 immediately, free it -> low peak RAM
    labs=[]; parts=[]; CH=150000
    for f in files:
        for d in pd.read_csv(f,low_memory=False,chunksize=CH):
            d=d.rename(columns={cfg["label"]:"Label"}); lab=d["Label"].astype(str); d=d.drop(columns=["Label"])
            keep=~lab.isin(cfg["drop"]); d=d[keep]; lab=lab[keep]; d=d.reindex(columns=cols)
            arr=np.empty((len(d),len(cols)),np.float32)
            for jc,c_ in enumerate(cols):
                if c_ in catcols:
                    v=d[c_].astype(str); cm=catmap[c_]
                    for u in v.unique():
                        if u not in cm: cm[u]=len(cm)
                    arr[:,jc]=v.map(cm).astype(np.float32).values
                else:
                    arr[:,jc]=_hexnum(d[c_]).fillna(0.0).astype(np.float32).values
            parts.append(arr); labs+=lab.tolist(); del d,arr; gc.collect()
    X=np.concatenate(parts,0); del parts; gc.collect(); X[~np.isfinite(X)]=0.0
    lo=np.quantile(X,0.0001,0); hi=np.quantile(X,0.9999,0); np.clip(X,lo,hi,out=X)     # in-place clip (no copy)
    nun=(X!=X[0]).any(0); drop=set([cols[j] for j in range(len(cols)) if not nun[j]])|set(LEAKAGE)
    kj=[j for j,c_ in enumerate(cols) if c_ not in drop]; X=np.ascontiguousarray(X[:,kj]); feat=[cols[j] for j in kj]
    difs=[d for d in list(dict.fromkeys(cfg["IG"]+cfg["LIME"])) if d in feat]; labs=np.array(labs)
    if SUBSAMPLE:
        rng=np.random.default_rng(SEED); idx=[]
        for c_ in np.unique(labs):
            ii=np.where(labs==c_)[0]; idx+=list(rng.choice(ii,min(PER_CLASS_SUB,len(ii)),replace=False))
        idx=np.array(idx); rng.shuffle(idx); X=X[idx]; labs=labs[idx]
    o=dict(X=X,labels=labs,feat=feat,difs=difs); save("p0",o); gc.collect()
    P(f"[P0] clean {X.shape} classes {len(np.unique(labs))} DIFs {len(difs)} {'(SUBSAMPLE)' if SUBSAMPLE else '(FULL, chunked low-RAM)'}")
    return o


### 1.5 — PHASE 1 — target + oracle + CMR + DIFS

In [ ]:
# ===== CELL 4 — PHASE 1: target (CNN-1D/MLP) + oracle + CMR (cached) =====
if HAVE_TORCH:
    class CNN1D(nn.Module):
        def __init__(s,d,k):
            super().__init__()
            s.c=nn.Sequential(nn.Conv1d(1,64,3,padding=1),nn.BatchNorm1d(64),nn.ReLU(),nn.MaxPool1d(2),
                              nn.Conv1d(64,128,3,padding=1),nn.BatchNorm1d(128),nn.ReLU(),nn.AdaptiveMaxPool1d(1))
            s.f=nn.Sequential(nn.Flatten(),nn.Linear(128,128),nn.ReLU(),nn.Dropout(0.3),nn.Linear(128,k))
        def forward(s,x): return s.f(s.c(x.unsqueeze(1)))
def phase1(force=False):
    if has("p1") and not force: P("[P1] cached"); return load("p1")
    d=phase0(); X,labels,feat,difs=d["X"],d["labels"],d["feat"],d["difs"]
    mean_=X.mean(0); scale_=X.std(0); scale_[scale_<1e-8]=1; Xs=((X-mean_)/scale_).astype(np.float32)
    del X; gc.collect()
    le=LabelEncoder().fit(labels); y=le.transform(labels); CLS=list(le.classes_); NORMAL=le.transform([BEN])[0]
    Xtr,Xte,ytr,yte=train_test_split(Xs,y,test_size=0.25,stratify=y,random_state=SEED)
    kind="mlp"
    if TARGET=="cnn" and HAVE_TORCH:
        kind="cnn"; rng=np.random.default_rng(SEED); idx=[]
        for c in range(len(CLS)):
            ii=np.where(ytr==c)[0]; idx+=list(rng.choice(ii,min(CNN_PER_CLASS,len(ii)),replace=len(ii)<CNN_PER_CLASS))
        idx=np.array(idx); rng.shuffle(idx); Xt=torch.tensor(Xtr[idx]); yt=torch.tensor(ytr[idx]); m=CNN1D(len(feat),len(CLS))
        opt=torch.optim.Adam(m.parameters(),1e-3); ce=nn.CrossEntropyLoss(); m.train()
        for ep in range(CNN_EPOCHS):
            pm=torch.randperm(len(Xt))
            for i in range(0,len(Xt),256):
                b=pm[i:i+256]; opt.zero_grad(); ce(m(Xt[b]),yt[b]).backward(); opt.step()
        m.eval(); torch.save(m.state_dict(),f"{CACHE}/cnn.pt")
        def proba(Z):
            if len(Z)==0: return np.zeros((0,len(CLS)),np.float32)
            with torch.no_grad(): return np.concatenate([torch.softmax(m(torch.tensor(np.ascontiguousarray(Z[i:i+8192]))),1).numpy() for i in range(0,len(Z),8192)])
    else:
        m=MLPClassifier(hidden_layer_sizes=(64,),max_iter=60,alpha=1e-2,random_state=SEED).fit(Xtr,ytr); cl=list(m.classes_)
        pickle.dump(m,open(f"{CACHE}/mlp.pkl","wb"))
        def proba(Z):
            Pr=m.predict_proba(Z.astype(np.float32)); full=np.zeros((len(Z),len(CLS)),np.float32)
            for j,c in enumerate(cl): full[:,c]=Pr[:,j]
            return full
    acc=float((proba(Xte).argmax(1)==yte).mean())
    from sklearn.metrics import f1_score; f1=f1_score(yte,proba(Xte).argmax(1),average="macro")
    P(f"[P1] target={kind} acc={acc:.4f} macroF1={f1:.4f}")
    si=np.random.default_rng(1).choice(len(Xtr),min(30000,len(Xtr)),replace=False)
    oracle=HistGradientBoostingClassifier(max_iter=150,random_state=1).fit(Xtr[si],ytr[si])
    fidx={f:feat.index(f) for f in feat}; cmr={}
    for c in [x for x in CLS if x!=BEN]:
        ci=le.transform([c])[0]; Xc=Xtr[ytr==ci]
        if len(Xc)<60: continue
        pool=[f for f in feature_pool(feat,difs) if f not in IMMUTABLE and Xc[:,fidx[f]].std()>=0.05]
        if not pool: continue
        pidx=[fidx[f] for f in pool]; Xcs=Xc[np.random.default_rng(0).choice(len(Xc),min(20000,len(Xc)),replace=False)]
        km=KMeans(n_clusters=1,n_init=3,random_state=SEED).fit(Xcs[:,pidx]); dd=cdist(Xcs[:,pidx],km.cluster_centers_).min(1)
        cmr[c]=dict(pidx=pidx,centroids=km.cluster_centers_,eps=float(np.percentile(dd,CMR_PCT)))
    gmin,gmax=Xs.min(0),Xs.max(0); grng=np.where((gmax-gmin)>1e-9,gmax-gmin,1).astype(np.float32)
    o=dict(kind=kind,oracle=oracle,cmr=cmr,Xtr=Xtr,Xte=Xte,ytr=ytr,yte=yte,le=le,CLS=CLS,NORMAL=NORMAL,
           feat=feat,difs=difs,mean_=mean_,scale_=scale_,gmin=gmin,gmax=gmax,grng=grng,fidx=fidx,acc=acc,f1=f1,
           ncls=len(CLS),nfeat=len(feat)); 
    # ---- reproducibility / methodology exports (previously only inside the pickle) ----
    try:
        # (1) ORACLE quality: Gate-2 depends entirely on this model, so report it
        from sklearn.metrics import f1_score as _f1s
        _op=oracle.predict(Xte)
        pd.DataFrame([dict(oracle="HistGradientBoosting",
                           accuracy=round(float((_op==yte).mean()),4),
                           macro_f1=round(float(_f1s(yte,_op,average="macro")),4),
                           n_train=int(len(si)))]).to_csv(f"{OUT}/oracle_metrics.csv",index=False)
        P(f"   [P1] oracle acc={float((_op==yte).mean()):.4f} macroF1={_f1s(yte,_op,average='macro'):.4f}")
    except Exception as _e: P("   [P1] oracle metrics skipped:",_e)
    try:
        # (2) CMR radius per class (the functionality manifold parameter)
        _cm=[[CLS[c] if isinstance(c,int) else c, len(v["pidx"]), int(len(v["centroids"])), round(float(v["eps"]),4)]
             for c,v in cmr.items()]
        pd.DataFrame(_cm,columns=["class","n_proj_features","n_centroids","cmr_eps"]).to_csv(f"{OUT}/cmr_eps.csv",index=False)
    except Exception as _e: P("   [P1] cmr export skipped:",_e)
    try:
        # (3) dataset summary + class distribution
        _u,_ct=np.unique(ytr,return_counts=True); _u2,_ct2=np.unique(yte,return_counts=True)
        _d=dict(zip(_u.tolist(),_ct.tolist())); _d2=dict(zip(_u2.tolist(),_ct2.tolist()))
        pd.DataFrame([[CLS[i],_d.get(i,0),_d2.get(i,0)] for i in range(len(CLS))],
                     columns=["class","train_rows","test_rows"]).to_csv(f"{OUT}/dataset_summary.csv",index=False)
    except Exception as _e: P("   [P1] dataset summary skipped:",_e)
    try:
        # (4) run configuration (every knob that changes the numbers)
        _cfgrows=[("DATASET",DATASET),("EVAL_CAP",EVAL_CAP),("generators","|".join(GENERATORS)),
                  ("GATE1_THR",GATE1_THR),("GATE2_theta",0.50),("GATE2_CMR",GATE2_CMR),
                  ("GATE3_operating",GATE3_OP),("HARD_SD_K",HARD_SD_K),("CMR_PCT",CMR_PCT),
                  ("n_features",len(feat)),("n_DIFs",len(difs)),("n_classes",len(CLS))]
        pd.DataFrame(_cfgrows,columns=["setting","value"]).to_csv(f"{OUT}/run_config.csv",index=False)
    except Exception as _e: P("   [P1] run config skipped:",_e)
    save("p1",o); P("[P1] cached"); return o

def helpers(S):
    NORMAL=S["NORMAL"]; grng=S["grng"]
    if S["kind"]=="cnn":
        m=CNN1D(S["nfeat"],S["ncls"]); m.load_state_dict(torch.load(f"{CACHE}/cnn.pt")); m.eval()
        def proba(Z):
            if len(Z)==0: return np.zeros((0,S["ncls"]),np.float32)
            with torch.no_grad(): return np.concatenate([torch.softmax(m(torch.tensor(np.ascontiguousarray(Z[i:i+8192]))),1).numpy() for i in range(0,len(Z),8192)])
    else:
        m=pickle.load(open(f"{CACHE}/mlp.pkl","rb")); cl=list(m.classes_)
        def proba(Z):
            Pr=m.predict_proba(Z.astype(np.float32)); full=np.zeros((len(Z),S["ncls"]),np.float32)
            for j,c in enumerate(cl): full[:,c]=Pr[:,j]
            return full
    pN=lambda Z: proba(Z)[:,NORMAL]; pred=lambda Z: proba(Z).argmax(1)
    L1=lambda Z,X0,mut:(np.abs(Z[:,mut]-X0[:len(Z),mut])/grng[mut]).sum(1)
    L0=lambda Z,X0,mut:(np.abs(Z[:,mut]-X0[:len(Z),mut])>1e-4).sum(1)
    return pN,pred,L1,L0
def bounds(S,c,sd_k=None):
    sd_k=HARD_SD_K if sd_k is None else sd_k
    le,Xtr,ytr,feat,difs,fidx,gmin,gmax=S["le"],S["Xtr"],S["ytr"],S["feat"],S["difs"],S["fidx"],S["gmin"],S["gmax"]
    ci=le.transform([c])[0]; mm=ytr==ci; mu=Xtr[mm].mean(0); sd=Xtr[mm].std(0)
    lo=(mu-sd_k*sd).astype(np.float32); hi=(mu+sd_k*sd).astype(np.float32)
    mut=[fidx[f] for f in feature_pool(feat,difs) if f not in IMMUTABLE and sd[fidx[f]]>=0.05]
    for f in craft:
        if f in fidx:
            j=fidx[f]; mut.append(j) if j not in mut else None; lo[j]=gmin[j]; hi[j]=gmax[j]
    return ci,mut,lo,hi


### 1.6 — PHASE 2 — generate candidates

In [ ]:
# ===== CELL 5 — PHASE 2: generate candidates (cached, incremental) =====
def phase2(gens=None, force=False):
    gens=gens or GENERATORS
    if force and has("p2"): os.remove(f"{CACHE}/p2.pkl")
    S=phase1(); pN,pred,L1,L0=helpers(S); oracle=S["oracle"]
    le,Xte,yte,Xtr,ytr=S["le"],S["Xte"],S["yte"],S["Xtr"],S["ytr"]
    cache=load("p2") if has("p2") else {}; rng=np.random.default_rng(SEED); t=time.time()
    for g in gens:
        if g in HEAVY and not ENABLE_HEAVY: P(f"   {g} skipped (ENABLE_HEAVY=False)"); continue
        gf=GENS[g]
        for c in [x for x in le.classes_ if x!=BEN]:
            if (c,g) in cache: continue
            ci,mut,lo,hi=bounds(S,c); te=Xte[yte==ci]
            if len(te)==0 or not mut: continue
            te=te[rng.choice(len(te),min(EVAL_CAP,len(te)),replace=False)]; det=te[pred(te)==ci]
            if len(det)<10: continue
            tr=Xtr[ytr==ci]; tr=tr[rng.choice(len(tr),min(1200,len(tr)),replace=False)]
            out=(gf(det,tr,lo,hi,mut,pN,L1,rng) if g in NEEDS_TR else gf(det,lo,hi,mut,pN,L1,rng))
            Xa=out[0] if isinstance(out,tuple) else out; q=out[1] if isinstance(out,tuple) else 0
            cmd=cdist(Xa[:,S["cmr"][c]["pidx"]],S["cmr"][c]["centroids"]).min(1)<=S["cmr"][c]["eps"] if c in S["cmr"] else np.ones(len(Xa),bool)
            cache[(c,g)]=dict(ci=ci,mut=mut,det=det,Xa=Xa.astype(np.float32),pN=pN(Xa),Pc=oracle.predict_proba(Xa)[:,ci],
                              l1=L1(Xa,det,mut),l0=L0(Xa,det,mut),cm=cmd,q=q)
        save("p2",cache); P(f"   {g} done t={time.time()-t:.0f}s (cached {len(cache)} blocks)")
    return cache


### 1.7 — PHASE 3 — tri-gate + survivors (+ result tables)

In [ ]:
# ===== CELL 6 — PHASE 3: Tri-Gate (re-runnable) + EXTRACT survivors =====
_THP={}
def _theta(S,ci,mode,fix):
    if mode=="fixed": return fix
    if ci not in _THP: _THP[ci]=max(0.5,float(np.percentile(S["oracle"].predict_proba(S["Xtr"][S["ytr"]==ci][:5000])[:,ci],10)))
    return _THP[ci]
def phase3(theta_mode=None,theta_fix=None,gate2_cmr=None,gate3=None,extract=True,save_name="survivors"):
    theta_mode=THETA_MODE if theta_mode is None else theta_mode
    theta_fix =THETA_FIX  if theta_fix  is None else theta_fix
    gate2_cmr =GATE2_CMR  if gate2_cmr  is None else gate2_cmr
    gate3     =GATE3_OP   if gate3      is None else gate3
    S=phase1(); cache=phase2()
    flds=[f for f in feature_pool(S["feat"],S["difs"]) if f in S["fidx"]]; recs=[]; percl={}
    perpair=[]; SWEEP=list(G3_SWEEP); sweep_rows=[]   # single-source tables for the figures
    for (c,g),D in cache.items():
        th=_theta(S,D["ci"],theta_mode,theta_fix); g1=D["pN"]>=GATE1_THR; orc=D["Pc"]>=th
        g2=(orc&D["cm"]) if gate2_cmr else orc; g12=g1&g2; keep=np.where(g12&(D["l1"]<gate3))[0]
        percl[c]=percl.get(c,0)+len(keep)
        perpair.append([c,g,len(D["Xa"]),int(g1.sum()),int(g12.sum()),len(keep),
                        round(100*float(g1.mean()),2),round(float(np.mean(D["l1"][keep])) if len(keep) else 0.0,4),
                        int(D.get("q",0)) if not hasattr(D.get("q",0),"__len__") else int(np.sum(D["q"]))])
        sweep_rows.append([c,g]+[int((g12&(D["l1"]<b)).sum()) for b in SWEEP])
        if extract:
            for i in keep:
                o=D["Xa"][i]*S["scale_"]+S["mean_"]
                r=dict(cls=c,gen=g,P_Normal=round(float(D["pN"][i]),3),oracle_Pc=round(float(D["Pc"][i]),3),
                       L1=round(float(D["l1"][i]),4),L0=int(D["l0"][i]))
                for f in flds: r[f]=round(float(o[S["fidx"][f]]),4)
                recs.append(r)
    tot=sum(percl.values())
    P(f"[P3] theta={theta_mode}({theta_fix}) Gate2={'oracle&CMR' if gate2_cmr else 'oracle'} L1<{gate3} -> FVAS={tot}  { {k:v for k,v in percl.items() if v>0} }")
    pd.DataFrame(perpair,columns=["class","gen","n","G1_evade","G1G2_func","FVAS","ASR_toNormal%","L1_mean","queries"]).to_csv(f"{OUT}/perclass_gate.csv",index=False)
    pd.DataFrame(sweep_rows,columns=["class","gen"]+[f"FVAS<{b}" for b in SWEEP]).to_csv(f"{OUT}/gate3_L1_sweep.csv",index=False)
    surv=pd.DataFrame(recs)
    if extract and len(surv): surv.to_csv(f"{OUT}/{save_name}.csv",index=False); P(f"     survivors -> {OUT}/{save_name}.csv ({len(surv)} rows)")

    # ================= additional CBAG analyses =================
    # (1) REACHABILITY: FVAS vs L1 budget under the OPERATING gates vs STRICT gates
    #     (strict = Gate-2 also requires the CMR functionality manifold)
    _reach=[]
    for b in SWEEP:
        _mod=0; _strict=0
        for (c,g),D in cache.items():
            _g1=D["pN"]>=GATE1_THR; _orc=D["Pc"]>=theta_fix
            _mod    += int((_g1 & _orc            & (D["l1"]<b)).sum())
            _strict += int((_g1 & _orc & D["cm"]  & (D["l1"]<b)).sum())
        _reach.append([b,_mod,_strict])
    pd.DataFrame(_reach,columns=["L1_budget","FVAS_operating","FVAS_strict_CMR"]).to_csv(f"{OUT}/T3_reachability.csv",index=False)

    # (2) RANGE TIGHTENING: how much narrower is the region survivors actually occupy
    #     than the region the rulebook ALLOWS?  rho<1 => survivors concentrate (CMR evidence)
    _A6=[]; _T5=[]
    for c in sorted({cc for (cc,_g) in cache.keys()}):
        _sub=surv[surv.cls==c] if len(surv) and "cls" in surv.columns else None
        if _sub is None or len(_sub)==0: continue
        try: _ci,_mut,_lo,_hi=bounds(S,c)
        except Exception:
            try: _ci,_mut,_lo,_hi=bounds(c)
            except Exception: continue
        _rhos=[]
        for j in _mut:
            _f=S["feat"][j]
            if _f not in _sub.columns: continue
            _raw_lo=float(_lo[j]*S["scale_"][j]+S["mean_"][j]); _raw_hi=float(_hi[j]*S["scale_"][j]+S["mean_"][j])
            _sv=_sub[_f].values.astype(float)
            if len(_sv)==0: continue
            _s_lo,_s_hi=float(_sv.min()),float(_sv.max())
            _rw=max(_raw_hi-_raw_lo,1e-9); _sw=max(_s_hi-_s_lo,0.0)
            _rho=_sw/_rw; _rhos.append(_rho)
            _A6.append([c,_f,round(_raw_lo,4),round(_raw_hi,4),round(_s_lo,4),round(_s_hi,4),
                        round(_rho,4),round(1.0/_rho,2) if _rho>1e-9 else None])
        if _rhos:
            _T5.append([c,len(_sub),len(_rhos),round(float(np.median(_rhos)),4),
                        round(float(np.mean(np.array(_rhos)<0.5)),3)])
    if _A6: pd.DataFrame(_A6,columns=["class","feature","raw_lo","raw_hi","surv_lo","surv_hi","rho","tightening_x"]).to_csv(f"{OUT}/A6_ranges.csv",index=False)
    if _T5:
        pd.DataFrame(_T5,columns=["class","n_surv","n_feat","median_rho","frac_rho<0.5"]).to_csv(f"{OUT}/T5_tightening.csv",index=False)
        P(f"     tightening -> {OUT}/T5_tightening.csv  (median rho per class; rho<1 = survivors concentrate)")

    # (3) SURVIVOR ROWS at additional budgets (not just counts)
    for _b in SWEEP:
        if _b==gate3: continue
        _r2=[]
        for (c,g),D in cache.items():
            _g1=D["pN"]>=GATE1_THR; _orc=D["Pc"]>=theta_fix
            _g2=(_orc&D["cm"]) if gate2_cmr else _orc
            for i in np.where(_g2&_g1&(D["l1"]<_b))[0][:5000]:
                _o=D["Xa"][i]*S["scale_"]+S["mean_"]
                _rr=dict(cls=c,gen=g,P_Normal=round(float(D["pN"][i]),3),oracle_Pc=round(float(D["Pc"][i]),3),
                         L1=round(float(D["l1"][i]),4),L0=int(D["l0"][i]))
                for f in flds: _rr[f]=round(float(_o[S["fidx"][f]]),4)
                _r2.append(_rr)
        if _r2: pd.DataFrame(_r2).to_csv(f"{OUT}/survivors_budget_{_b}.csv",index=False)
    P(f"     reachability + tightening + multi-budget survivors written to {OUT}")

    # (4) CONSTRAINED (CBAG) vs UNCONSTRAINED BASELINE attacks -- the core CBAG claim:
    #     baselines may evade as well or better, but their outputs are largely NON-FUNCTIONAL.
    _rows=[]
    for g in sorted({gg for (_c,gg) in cache.keys()}):
        _n=_e=_f=_s=0; _q=0
        for (c,gg),D in cache.items():
            if gg!=g: continue
            _g1=D["pN"]>=GATE1_THR; _orc=D["Pc"]>=theta_fix
            _n+=len(D["pN"]); _e+=int(_g1.sum()); _f+=int((_g1&_orc).sum())
            _s+=int((_g1&_orc&(D["l1"]<gate3)).sum()); _q+=int(D["q"])
        _rows.append([g, GEN_CLASS.get(g,"constrained"), _n, _e, _f, _s,
                      round(100*_e/max(_n,1),2), round(100*_f/max(_e,1),2), round(100*_s/max(_n,1),3),
                      round(1000*_s/max(_q,1),3)])
    _B=pd.DataFrame(_rows,columns=["gen","type","candidates","evaded","functional","FVAS",
                                   "evasion%","functional%_of_evaders","FVAS%","FVAS_per_1k_queries"])
    _B.to_csv(f"{OUT}/T7_constrained_vs_baseline.csv",index=False)

    # (5) PERTURBATION DISTRIBUTION per generator (stealth is a distributional claim, not a mean)
    _pd_rows=[]
    for g in sorted({gg for (_c,gg) in cache.keys()}):
        _l1=[]; _l0=[]; _l1s=[]
        for (c,gg),D in cache.items():
            if gg!=g: continue
            _l1.append(np.asarray(D["l1"],float)); _l0.append(np.asarray(D["l0"],float))
            _k=(D["pN"]>=GATE1_THR)&(D["Pc"]>=theta_fix)&(D["l1"]<gate3)
            if _k.any(): _l1s.append(np.asarray(D["l1"],float)[_k])
        if not _l1: continue
        _l1=np.concatenate(_l1); _l0=np.concatenate(_l0)
        _l1s=np.concatenate(_l1s) if _l1s else np.array([np.nan])
        _pd_rows.append([g,GEN_CLASS.get(g,"constrained"),
                         *[round(float(np.percentile(_l1,p)),4) for p in (10,25,50,75,90)],
                         round(float(np.mean(_l0)),2), round(float(np.median(_l0)),1),
                         round(float(np.nanmedian(_l1s)),4)])
    pd.DataFrame(_pd_rows,columns=["gen","type","L1_p10","L1_p25","L1_p50","L1_p75","L1_p90",
                                   "L0_mean","L0_median","L1_median_survivors"]
                 ).to_csv(f"{OUT}/T8_perturbation_dist.csv",index=False)

    # (6) PER-FEATURE perturbation frequency: which mutable DIFs does the attacker actually move?
    #     (validates the DIFS -> CBAG link)
    _ff={}
    for (c,g),D in cache.items():
        _k=np.where((D["pN"]>=GATE1_THR)&(D["Pc"]>=theta_fix)&(D["l1"]<gate3))[0]
        if len(_k)==0: continue
        _d=np.abs(D["Xa"][_k]-D["det"][_k])
        for j in D["mut"]:
            _f=S["feat"][j]; _moved=float((_d[:,j]>1e-6).mean()); _mag=float(_d[:,j].mean())
            a=_ff.setdefault((c,_f),[0,0.0,0])
            a[0]+=len(_k); a[1]+=_mag*len(_k); a[2]+=int((_d[:,j]>1e-6).sum())
    if _ff:
        _fr=[[c,f,v[0],round(v[2]/max(v[0],1),3),round(v[1]/max(v[0],1),4)] for (c,f),v in _ff.items()]
        pd.DataFrame(_fr,columns=["class","feature","n_survivors","frac_moved","mean_abs_delta"]
                     ).sort_values(["class","frac_moved"],ascending=[True,False]
                     ).to_csv(f"{OUT}/T9_feature_perturbation.csv",index=False)
    P(f"     constrained-vs-baseline + perturbation dist + feature frequency written to {OUT}")

    return tot,surv


### ▶ Run SECTION 1

In [ ]:
phase0() if 'phase0' in dir() else None
phase1(); phase2()
phase3()
print('SECTION 1 done ->', OUT)

# SECTION 2 — SBAR (held-out cross-generator, AccessPoint)

### 2.1 — SBAR config (paper-grade; pause/resume)

In [ ]:
# ===== PAPER-GRADE CONFIGURATION (AccessPoint) =====
#   5 seeds | all 9 generators | all non-held-out generators used for training
#   eval classes auto-selected then FROZEN | LOGO = leave-one-generator-out ENABLED
#   DIF pool switches (USE_XAI_DIFS / DIF_MODE) left as configured in SECTION 1
#   NOTE: LOGO multiplies runtime by the number of generators - expect a long run.
# =====================================
import os,pickle,json,time,warnings,numpy as np; warnings.filterwarnings("ignore")
import torch,torch.nn as nn; torch.set_num_threads(os.cpu_count() or 1)
from scipy.stats import wilcoxon
try:
    ids_metrics
except NameError:
    from sklearn.metrics import precision_recall_fscore_support, balanced_accuracy_score, matthews_corrcoef
    def ids_metrics(y_true,y_pred,normal_idx,class_names):
        import numpy as _np, pandas as _pd
        y_true=_np.asarray(y_true); y_pred=_np.asarray(y_pred); K=len(class_names)
        Pr,Re,Ff,Su=precision_recall_fscore_support(y_true,y_pred,labels=list(range(K)),zero_division=0)
        atk=[i for i in range(K) if i!=normal_idx]; DR=float(_np.mean([Re[i] for i in atk])) if atk else float('nan')
        nn_=int((y_true==normal_idx).sum()); fp=int(((y_true==normal_idx)&(y_pred!=normal_idx)).sum())
        return dict(accuracy=float((y_true==y_pred).mean()),macro_f1=float(_np.mean(Ff)),
                    weighted_f1=float(_np.average(Ff,weights=Su)) if Su.sum() else float('nan'),
                    balanced_acc=float(balanced_accuracy_score(y_true,y_pred)),mcc=float(matthews_corrcoef(y_true,y_pred)),
                    detection_rate=DR,fpr=fp/max(nn_,1)), _pd.DataFrame({'class':class_names,'precision':Pr,'recall':Re,'f1':Ff,'support':Su.astype(int)})
def P(*a): print(*a,flush=True)
CACHE=f"cbag_cache_{DATASET}"; RES="SBAR_results_AccessPoint"; CKPT=f"{RES}/holdout13_rows.json"; os.makedirs(RES,exist_ok=True)
# ---- HOLD-OUT DESIGN ----
# families let you withhold a whole ATTACK TYPE (much stronger generalisation claim
# than withholding one metaheuristic while training on three sibling metaheuristics)
GEN_FAMILY = {"C-SA":"metaheuristic","C-PSO":"metaheuristic","C-GA":"metaheuristic","C-RL":"metaheuristic",
              "C-BO":"bayesian","CG-Diffusion":"learned","CC-CGAN":"learned",
              "HopSkipJump":"decision-based","Sign-OPT":"decision-based"}
HOLDOUT_FAMILY = None       # None = single-generator hold-out (HOLDOUT_GEN below).
                            # e.g. "learned" or "decision-based" => withhold that WHOLE family
HOLDOUT_GEN="C-PSO"                 # generator withheld from training, used ONLY to re-attack
TRAIN_GENS=None                     # None => auto: ALL cached generators EXCEPT the hold-out (recommended)
EVAL_CLS=None                       # None => auto-select once, then FROZEN for every config (comparable)
LOGO=True                           # PAPER: leave-one-generator-out (rotate hold-out over all generators).
                                    #   False = single fixed hold-out (fast; use for the proposal)
                                    #   True  = strongest evidence, but runtime x n_generators (for the paper)
LOGO_CONFIGS=["Undefended","PGD-AT","Gentle-SBAR"]   # PAPER: headline pair + strongest baseline
# ---- RUN CONTROL (pause / resume) ----
RUN_SBAR      = True      # False => skip SBAR entirely (CBAG-only session)
MAX_RUNS      = None      # e.g. 12 => do at most 12 (config,seed) runs this execution, then PAUSE.
                          #   Re-running the cell RESUMES from the checkpoint. None = run to completion.
MAX_MINUTES   = None      # e.g. 45 => stop cleanly after ~45 min. None = no time limit.
SEEDS=[42,7,123,2024,31]      # PAPER: 5 seeds (proposal used 3)
BUD=0.25; EPOCHS=3; CLEAN_PC=1500   # BUD = SBAR training budget for AccessPoint (looser than the 0.10 operating point)
IMMUTABLE={'ip.proto','ip.ttl','ip.dst','id.orig_h','id.resp_h'}
UNSETTABLE={'radiotap.dbm_antsignal','wlan_radio.signal_dbm','radiotap.datarate','radiotap.channel.freq','wlan_radio.frequency','radiotap.channel.flags.ofdm'}
craft=['wlan.fc.retry','wlan.fcs.bad_checksum','wlan.rsn.capabilities.mfpc','frame.encap_type']
def fpool(feat,difs): return [f for f in difs if f not in UNSETTABLE]
G1_THR_SBAR = GATE1_THR      # use THIS dataset's Gate-1 threshold everywhere in SBAR


### 2.2 — load caches + auto TRAIN_GENS/EVAL_CLS

In [ ]:
S=pickle.load(open(f"{CACHE}/p1.pkl","rb")); cache=pickle.load(open(f"{CACHE}/p2.pkl","rb"))
Xtr,Xte,ytr,yte,le,CLS,NORMAL=S["Xtr"],S["Xte"],S["ytr"],S["yte"],S["le"],S["CLS"],S["NORMAL"]
nfeat,ncls,fidx,feat,difs,gmin,gmax,grng=S["nfeat"],S["ncls"],S["fidx"],S["feat"],S["difs"],S["gmin"],S["gmax"],S["grng"]
class CNN1D(nn.Module):
    def __init__(s,d,k):
        super().__init__(); s.c=nn.Sequential(nn.Conv1d(1,64,3,padding=1),nn.BatchNorm1d(64),nn.ReLU(),nn.MaxPool1d(2),
            nn.Conv1d(64,128,3,padding=1),nn.BatchNorm1d(128),nn.ReLU(),nn.AdaptiveMaxPool1d(1))
        s.f=nn.Sequential(nn.Flatten(),nn.Linear(128,128),nn.ReLU(),nn.Dropout(0.3),nn.Linear(128,k))
    def forward(s,x): return s.f(s.c(x.unsqueeze(1)))
def base(): m=CNN1D(nfeat,ncls); m.load_state_dict(torch.load(f"{CACHE}/cnn.pt")); return m
def proba(m,Z):
    m.eval()
    with torch.no_grad(): return np.concatenate([torch.softmax(m(torch.tensor(np.ascontiguousarray(Z[i:i+8192]))),1).numpy() for i in range(0,len(Z),8192)])
def mkpn(m): return (lambda Z:proba(m,Z)[:,NORMAL]),(lambda Z:proba(m,Z).argmax(1))
L1=lambda Z,X0,mut:(np.abs(Z[:,mut]-X0[:len(Z),mut])/grng[mut]).sum(1)
def bounds(c,sd_k=3.0):
    ci=le.transform([c])[0]; mm=ytr==ci; mu=Xtr[mm].mean(0); sd=Xtr[mm].std(0)
    lo=(mu-sd_k*sd).astype(np.float32); hi=(mu+sd_k*sd).astype(np.float32)
    mut=[fidx[f] for f in fpool(feat,difs) if f not in IMMUTABLE and sd[fidx[f]]>=0.05]
    for f in craft:
        if f in fidx: j=fidx[f]; mut.append(j) if j not in mut else None; lo[j]=gmin[j]; hi[j]=gmax[j]
    return ci,mut,lo,hi

# --- resolve TRAIN_GENS / EVAL_CLS from the ACTUAL cache (works with any DIF pool / generator roster) ---
_cached_gens=sorted({g for (_c,g) in cache.keys()})
if TRAIN_GENS is None: TRAIN_GENS={g for g in _cached_gens if g!=HOLDOUT_GEN}
if HOLDOUT_GEN not in _cached_gens:
    print(f"[SBAR] WARNING: hold-out {HOLDOUT_GEN} not in cache {_cached_gens}; using the last generator as hold-out")
    HOLDOUT_GEN=_cached_gens[-1]; TRAIN_GENS={g for g in _cached_gens if g!=HOLDOUT_GEN}
if EVAL_CLS is None:
    _surv={}
    for (c,g),D in cache.items():
        _surv[c]=_surv.get(c,0)+int(((D["pN"]>=G1_THR_SBAR)&(D["Pc"]>=0.5)&(D["l1"]<BUD)).sum())
    EVAL_CLS=[c for c,n in sorted(_surv.items(),key=lambda kv:-kv[1]) if n>0 and c!=le.classes_[NORMAL]][:6]
    if not EVAL_CLS: EVAL_CLS=[c for c in le.classes_ if c!=le.classes_[NORMAL]][:5]
print(f"[SBAR] generators cached={_cached_gens}")
# --- hold-out family support: withhold an entire attack type ---
if HOLDOUT_FAMILY is not None:
    _fam=[g for g in _cached_gens if GEN_FAMILY.get(g)==HOLDOUT_FAMILY]
    if _fam:
        TRAIN_GENS={g for g in _cached_gens if g not in _fam}
        # attack with the most productive member of the withheld family
        _cnt={g:sum(int(((D["pN"]>=G1_THR_SBAR)&(D["Pc"]>=0.5)&(D["l1"]<BUD)).sum())
                    for (c,gg),D in cache.items() if gg==g) for g in _fam}
        HOLDOUT_GEN=max(_cnt,key=_cnt.get)
        print(f"[SBAR] hold-out FAMILY='{HOLDOUT_FAMILY}' -> withheld {_fam}; attacking with {HOLDOUT_GEN}")
    else:
        print(f"[SBAR] WARNING: no cached generators in family '{HOLDOUT_FAMILY}'; keeping single hold-out")
print(f"[SBAR] hold-out={HOLDOUT_GEN} | train on={sorted(TRAIN_GENS)}")
EVAL_CLS=list(EVAL_CLS)   # FROZEN: identical class set for every config so results are comparable
print(f"[SBAR] eval classes (frozen for all configs)={EVAL_CLS}")


### 2.3 — held-out attack

In [ ]:
def C_PSO_attack(det,lo,hi,mut,pN,rng,wp=0.3,Pn=12,iters=40):
    N,d,F=len(det),len(mut),det.shape[1]; b=np.repeat(det[:,None,:],Pn,1).astype(np.float32)
    b[:,:,mut]=rng.uniform(lo[mut],hi[mut],(N,Pn,d)).astype(np.float32); vel=np.zeros((N,Pn,d),np.float32)
    en=lambda B:-pN(B.reshape(-1,F)).reshape(N,Pn)+wp*(np.abs(B[:,:,mut]-det[:,None,mut])/(hi[mut]-lo[mut]+1e-9)).sum(2)
    E=en(b); pb=b.copy(); pbE=E.copy(); gi=pbE.argmin(1); gb=pb[np.arange(N),gi].copy()
    for _ in range(iters):
        r1=rng.random((N,Pn,d),dtype=np.float32); r2=rng.random((N,Pn,d),dtype=np.float32)
        vel=0.6*vel+1.5*r1*(pb[:,:,mut]-b[:,:,mut])+1.5*r2*(gb[:,None,mut]-b[:,:,mut])
        b[:,:,mut]=np.clip(b[:,:,mut]+vel,lo[mut],hi[mut]); E=en(b)
        imp=E<pbE; pb[imp]=b[imp]; pbE[imp]=E[imp]; gi=pbE.argmin(1); gb=pb[np.arange(N),gi].copy()
    return gb

# ---- HOLD-OUT ATTACK DISPATCH -------------------------------------------------
# The re-attack MUST use the generator that was actually withheld from training.
# (Previously this was hardcoded to PSO, which silently made any non-PSO hold-out
#  circular -- PSO would be in the training set AND be the attacker.)
def holdout_attack(det, lo, hi, mut, pN, rng, ci=None):
    gname = HOLDOUT_GEN
    gf = GENS.get(gname) if "GENS" in globals() else None
    if gf is None:                       # generator fn not in scope -> safe fallback
        return C_PSO_attack(det, lo, hi, mut, pN, rng)
    try:
        if gname in NEEDS_TR:            # these generators need in-class training data
            tr = Xtr[ytr == ci] if ci is not None else Xtr[:512]
            if len(tr) < 20: return C_PSO_attack(det, lo, hi, mut, pN, rng)
            out = gf(det, tr, lo, hi, mut, pN, L1, rng)
        else:
            out = gf(det, lo, hi, mut, pN, L1, rng)
        Xa = out[0] if isinstance(out, tuple) else out
        Xa = np.asarray(Xa, np.float32)
        if Xa.ndim != 2 or len(Xa) != len(det): raise ValueError("bad shape")
        return Xa
    except Exception as e:
        print(f"[SBAR] hold-out attack '{gname}' failed ({type(e).__name__}); falling back to C-PSO")
        return C_PSO_attack(det, lo, hi, mut, pN, rng)


### 2.4 — training set

In [ ]:
def build_sets(seed):
    rng=np.random.default_rng(seed); ci=[]
    for c in range(ncls):
        ii=np.where(ytr==c)[0]; ci+=list(rng.choice(ii,min(CLEAN_PC,len(ii)),replace=len(ii)<CLEAN_PC))
    Xc,Yc=Xtr[np.array(ci)],ytr[np.array(ci)]
    aX=[];aY=[];aM=[];aG=[];uX=[];uY=[]
    for (c,g),D in cache.items():
        if g not in TRAIN_GENS: continue
        keep=np.where((D["pN"]>=G1_THR_SBAR)&(D["Pc"]>=0.5)&(D["l1"]<BUD))[0]; ev=np.where(D["pN"]>=G1_THR_SBAR)[0]
        for i in keep: aX.append(D["Xa"][i]);aY.append(D["ci"]);aM.append(min(abs(G1_THR_SBAR-D["pN"][i]),0.1));aG.append(g)   # margin from this dataset's threshold
        for i in ev[:30]: uX.append(D["Xa"][i]);uY.append(D["ci"])
    aX=np.array(aX,np.float32);aY=np.array(aY);aM=np.array(aM,np.float32);aG=np.array(aG)
    uX=np.array(uX,np.float32);uY=np.array(uY)
    idx=rng.permutation(len(aX))[:int(0.7*len(aX))]
    return Xc,Yc,aX[idx],aY[idx],aM[idx],aG[idx],uX,uY

### 2.5 — weight schemes

In [ ]:
def wfun(M,mode):
    if mode=="none": return np.ones(len(M),np.float32)
    if mode=="full": return (1.0/(0.1-M+0.001)).astype(np.float32)
    if mode=="gentle": return (1.0+4.0*(M/0.1)).astype(np.float32)
def hn_full(M): return 1.0/(0.1-M+0.001)

### 2.6 — train

In [ ]:
def train(kind,seed):
    torch.manual_seed(seed); r=np.random.default_rng(seed)
    Xc,Yc,EX,EY,EM,EG,uX,uY=build_sets(seed)
    if kind=="Undefended": return base()
    if kind=="Naive-Aug":
        Xn=np.concatenate([Xc,Xc+r.normal(0,0.1,Xc.shape).astype(np.float32)]); Yn=np.concatenate([Yc,Yc]); W=np.ones(len(Xn),np.float32)
        return _fit(Xn,Yn,W,r)
    if kind=="PGD-AT":
        m0=base(); m0.eval(); Xa=torch.tensor(Xc[:6000].copy()); Xo=Xa.clone(); Xa.requires_grad_(True); ce=nn.CrossEntropyLoss()
        for _ in range(5):
            g,=torch.autograd.grad(ce(m0(Xa),torch.tensor(Yc[:6000])),Xa)
            with torch.no_grad(): Xa=torch.max(torch.min(Xa+0.03*g.sign(),Xo+0.3),Xo-0.3)
            Xa.requires_grad_(True)
        Xp=Xa.detach().numpy().astype(np.float32); return _fit(np.concatenate([Xc,Xp]),np.concatenate([Yc,Yc[:6000]]),np.ones(len(Xc)+6000,np.float32),r)
    # SBAR family: choose survivor subset + weight mode + mix
    frac=0.30; mode="gentle";
    if kind in("No-HardNeg",): mode="none"
    if kind in("Full-SBAR","No-Curriculum","EWC","Adaptive-Mix","Mix-15%","Mix-50%","GFF-SBAR","Intersection-SBAR"): mode="full"
    if kind in("No-Constraints","Gentle-SBAR"): mode="gentle" if kind=="Gentle-SBAR" else "none"
    if kind=="Mix-15%": frac=0.15
    if kind=="Mix-50%": frac=0.50
    if kind=="No-Constraints": EX,EY,EM=uX,uY,np.ones(len(uX),np.float32)*0.05
    if kind=="GFF-SBAR":
        bg=max(set(EG.tolist()),key=lambda g:(EG==g).sum()); mm=EG==bg; EX,EY,EM=EX[mm],EY[mm],EM[mm]
    if kind=="Intersection-SBAR":
        coreI=np.argsort(EM)[:max(20,len(EM)//3)]; EX,EY,EM=EX[coreI],EY[coreI],EM[coreI]
    nfv=int(frac/(1-frac)*len(Xc)); ei=r.integers(0,len(EX),nfv)
    XX=np.concatenate([Xc,EX[ei]]); YY=np.concatenate([Yc,EY[ei]]); W=np.concatenate([np.ones(len(Xc),np.float32),wfun(EM[ei],mode)])
    return _fit(XX,YY,W,r)
def _fit(XX,YY,W,r):
    m=base(); opt=torch.optim.Adam(m.parameters(),5e-4); ce=nn.CrossEntropyLoss(reduction="none"); m.train()
    for _ in range(EPOCHS):
        pm=r.permutation(len(XX))
        for i in range(0,len(XX),256):
            b=pm[i:i+256]; opt.zero_grad(); (ce(m(torch.tensor(XX[b])),torch.tensor(YY[b]))*torch.tensor(W[b])).mean().backward(); opt.step()
    m.eval(); return m

### 2.7 — evaluate

In [ ]:
def evaluate(m,seed):
    pN,pred=mkpn(m); rng=np.random.default_rng(seed)
    po=proba(m,Xte).argmax(1); clean=round(100*(po==yte).mean(),2); nm=yte==NORMAL; Normal=round(100*(po[nm]==NORMAL).mean(),2)
    _sm,_=ids_metrics(yte,po,NORMAL,list(le.classes_))
    macro_f1=round(100*_sm['macro_f1'],2); bal_acc=round(100*_sm['balanced_acc'],2); mcc=round(_sm['mcc'],4); DR=round(100*_sm['detection_rate'],2); FPR=round(100*_sm['fpr'],2)
    per={}; perF={}
    for c in EVAL_CLS:
        ci=le.transform([c])[0]; _,mut,lo,hi=bounds(c); te=Xte[yte==ci]
        if len(te)==0 or not mut: continue
        te=te[rng.choice(len(te),min(150,len(te)),replace=False)]; det=te[pred(te)==ci]
        if len(det)<10: per[c]=0.0; perF[c]=0.0; continue
        Xa=holdout_attack(det,lo,hi,mut,pN,rng,ci)
        _g1=pN(Xa)>=G1_THR_SBAR
        per[c]=round(100*_g1.mean(),2)
        # THREAT-MATCHED residual: the threat model requires a FUNCTIONAL, STEALTHY evasion (FVAS),
        # not bare evasion. Scoring only Gate-1 favours defences trained on unconstrained evaders.
        try:
            _pr=S["oracle"].predict_proba(Xa); _cl=list(S["oracle"].classes_)
            _g2=(_pr[:,_cl.index(ci)]>=0.5) if ci in _cl else np.ones(len(Xa),bool)
        except Exception:
            _g2=np.ones(len(Xa),bool)
        try:
            _g3=L1(Xa,det,mut)<GATE3_OP
        except Exception:
            _g3=np.ones(len(Xa),bool)
        perF[c]=round(100*(_g1&_g2&_g3).mean(),2)
    resid=round(float(np.mean(list(per.values()))),2) if per else float("nan")
    residF=round(float(np.mean(list(perF.values()))),2) if perF else float("nan")
    wb=[]
    for c in EVAL_CLS:
        ci=le.transform([c])[0]; _,mut,lo,hi=bounds(c); te=Xte[yte==ci][:150]
        if len(te)==0 or not mut: continue
        x=torch.tensor(te.copy()); x.requires_grad_(True)
        g,=torch.autograd.grad(nn.CrossEntropyLoss()(m(x),torch.full((len(te),),ci)),x)
        xa=te.copy(); mm=np.array(mut); xa[:,mm]=np.clip(te[:,mm]+0.2*np.sign(g.detach().numpy())[:,mm],lo[mm],hi[mm])
        wb.append(100*(pred(xa)!=ci).mean())
    return dict(clean=clean,Normal=Normal,macro_f1=macro_f1,bal_acc=bal_acc,mcc=mcc,DR=DR,FPR=FPR,residual=resid,residual_fvas=residF,whitebox=round(float(np.mean(wb)),2),per_class=per)

### 2.8 — configurations

In [ ]:
CONFIGS=["Undefended","Naive-Aug","PGD-AT","Gentle-SBAR","No-HardNeg","Full-SBAR","No-Curriculum","EWC","Adaptive-Mix","Mix-15%","Mix-50%","No-Constraints","GFF-SBAR","Intersection-SBAR"]
GROUP={"Undefended":"baseline","Naive-Aug":"baseline","PGD-AT":"baseline","Gentle-SBAR":"SBAR (recommended)","No-HardNeg":"ablation","Full-SBAR":"ablation","No-Curriculum":"ablation","EWC":"ablation","Adaptive-Mix":"ablation","Mix-15%":"ablation","Mix-50%":"ablation","No-Constraints":"ablation","GFF-SBAR":"variant","Intersection-SBAR":"variant"}

### 2.9 — run loop (+LOGO)

In [ ]:
def run():
    done=json.load(open(CKPT)) if os.path.exists(CKPT) else []
    have={(r["config"],r["seed"]) for r in done}
    total=len(CONFIGS)*len(SEEDS); t0=time.time(); n_new=0
    for seed in SEEDS:
        for k in CONFIGS:
            if (k,seed) in have: continue
            if MAX_RUNS is not None and n_new>=MAX_RUNS:
                P(f"[PAUSE] budget reached ({n_new} runs). {len(done)}/{total} done - re-run this cell to RESUME."); return done
            if MAX_MINUTES is not None and (time.time()-t0)/60>=MAX_MINUTES:
                P(f"[PAUSE] time limit reached. {len(done)}/{total} done - re-run this cell to RESUME."); return done
            t=time.time(); m=train(k,seed); e=evaluate(m,seed)
            done.append(dict(config=k,group=GROUP[k],seed=seed,**{x:e[x] for x in e if x!="per_class"},per_class=e["per_class"]))
            json.dump(done,open(CKPT,"w"),indent=1); n_new+=1
            P(f"{k:<18} s{seed}: clean {e['clean']} Normal {e['Normal']} resid {e['residual']} wb {e['whitebox']} ({time.time()-t:.0f}s)  [{len(done)}/{total}]")
    P(f"[DONE] {len(done)}/{total} runs complete.")
    return done

def run_logo():
    """Leave-one-generator-out: rotate the hold-out across every cached generator.
    Trains on all others, re-attacks with the held-out one. Strong evidence that the
    result is not an artifact of which generator happened to be withheld."""
    global HOLDOUT_GEN, TRAIN_GENS
    gens=sorted({g for (_c,g) in cache.keys()})
    ck=f"{RES}/logo_rows.json"
    rows=json.load(open(ck)) if os.path.exists(ck) else []
    have={(r["config"],r["seed"],r.get("holdout")) for r in rows}
    keep_h, keep_t = HOLDOUT_GEN, TRAIN_GENS
    try:
        for hg in gens:
            HOLDOUT_GEN=hg; TRAIN_GENS={g for g in gens if g!=hg}
            for seed in SEEDS:
                for k in LOGO_CONFIGS:
                    if (k,seed,hg) in have: continue
                    t=time.time(); m=train(k,seed); e=evaluate(m,seed)
                    rows.append(dict(config=k,group=GROUP[k],seed=seed,holdout=hg,
                                     **{x:e[x] for x in e if x!="per_class"},per_class=e["per_class"]))
                    json.dump(rows,open(ck,"w"),indent=1)
                    P(f"[LOGO hold-out={hg}] {k:<14} s{seed}: resid {e['residual']} clean {e['clean']} ({time.time()-t:.0f}s)")
    finally:
        HOLDOUT_GEN, TRAIN_GENS = keep_h, keep_t
    return rows

def aggregate_logo(rows=None):
    import pandas as pd
    ck=f"{RES}/logo_rows.json"
    rows=rows if rows is not None else (json.load(open(ck)) if os.path.exists(ck) else [])
    if not rows: P("[LOGO] no rows"); return None
    df=pd.DataFrame(rows)
    g=df.groupby(["holdout","config"]).agg(residual=("residual","mean"),residual_std=("residual","std"),
                                           clean=("clean","mean")).round(2).reset_index()
    g.to_csv(f"{RES}/LOGO_by_holdout.csv",index=False)
    piv=g.pivot(index="holdout",columns="config",values="residual")
    P("\n=== LOGO: residual ASR % by held-out generator ==="); P(piv.to_string())
    P("\nmean over hold-outs:"); P(piv.mean().round(2).to_string())
    return g


### 2.10 — aggregate

In [ ]:
def aggregate(rows):
    import pandas as pd
    from scipy.stats import wilcoxon
    df=pd.DataFrame(rows)
    order=["Undefended","Naive-Aug","PGD-AT","Gentle-SBAR","No-HardNeg","Full-SBAR","No-Curriculum","EWC","Adaptive-Mix","Mix-15%","Mix-50%","No-Constraints","GFF-SBAR","Intersection-SBAR"]
    # aggregate only the columns present (older checkpoints lack the IDS-metric fields)
    _spec={"group":("group","first"),"clean":("clean","mean"),"Normal":("Normal","mean"),
           "residual":("residual","mean"),"residual_std":("residual","std"),"whitebox":("whitebox","mean")}
    for _m in ["residual_fvas","macro_f1","bal_acc","mcc","DR","FPR"]:
        if _m in df.columns: _spec[_m]=(_m,"mean")
    g=df.groupby("config").agg(**_spec).reindex(order).round(2)
    und=g.loc["Undefended","residual"]; g["reduction_x"]=(und/g["residual"].replace(0,np.nan)).round(1)
    def n01(s,hi): s=s.astype(float); r=(s-s.min())/(s.max()-s.min()+1e-9); return r if hi else 1-r
    g["Balanced"]=(0.35*n01(g["residual"],False)+0.20*n01(g["whitebox"],False)+0.20*n01(g["clean"],True)+0.25*n01(g["Normal"],True)).round(3)
    g.to_csv(f"{RES}/E1_heldout_13configs.csv")
    try: g.reset_index().to_latex(f"{RES}/E1_heldout_13configs.tex",index=False,float_format="%.2f")
    except Exception: pass
    P("\n=== E1 held-out (13 configs, 3 seeds) ==="); P(g.to_string())
    # H3 per-class Undefended vs Gentle-SBAR
    perU={};perG={}
    for r in rows:
        if r["config"]=="Undefended":
            for c,v in r["per_class"].items(): perU.setdefault(c,[]).append(v)
        if r["config"]=="Gentle-SBAR":
            for c,v in r["per_class"].items(): perG.setdefault(c,[]).append(v)
    h3=[]
    for c in perU:
        u=np.mean(perU[c]); s=np.mean(perG.get(c,[0])); red=round(100*(1-s/u),1) if u>0 else float("nan")
        h3.append([c,round(u,2),round(s,2),red])
    pd.DataFrame(h3,columns=["class","ASR_undef%","ASR_Gentle%","reduction%"]).to_csv(f"{RES}/H3_heldout.csv",index=False)
    # Wilcoxon
    pu=[];pg=[]
    for r in rows:
        if r["config"]=="Undefended":
            gg=[x for x in rows if x["config"]=="Gentle-SBAR" and x["seed"]==r["seed"]]
            if gg:
                for c in r["per_class"]:
                    if c in gg[0]["per_class"]: pu.append(r["per_class"][c]); pg.append(gg[0]["per_class"][c])
    try:
        w,p=wilcoxon(pu,pg); P(f"Wilcoxon Undefended vs Gentle-SBAR (n={len(pu)}): W={w:.1f}, p={p:.5f}")
    except Exception as ex: P("Wilcoxon:",ex)
    return g

### ▶ Run SECTION 2

In [ ]:
if not RUN_SBAR:
    print("[SBAR] skipped (RUN_SBAR=False) - CBAG-only session")
else:
    rows=run()
    if len({(r["config"],r["seed"]) for r in rows})==len(CONFIGS)*len(SEEDS):
        aggregate(rows)
        if LOGO: aggregate_logo(run_logo())      # optional: leave-one-generator-out
    else:
        print("[SBAR] partial - re-run this cell to resume; aggregate runs when complete")


# SECTION 3 — All charts + tables → PDF + TikZ

### 3.1 — imports + shims + _save

In [ ]:
#!/usr/bin/env python3
# figures_all.py — COMPLETE dissertation chart suite for UAV1 (run it, then pick what you want).
# DIFS (quadrant + 4 feature-colour styles + heatmap + mutable/locked), CBAG (funnel + per-generator
# + per-class + FVR + class×gen heatmap + quadrant@CBAG + adaptive L1 curve + stacked contribution),
# SBAR (13-config panel + H3 + reduction ranking + weight curve).
# Writes FIG/<name>.png for each + FIG/ALL_RESULTS.pdf (everything, one file).
import os,json,numpy as np,pandas as pd,warnings; warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
# --- compat shims so tikzplotlib works on modern matplotlib/webcolors (best-effort) ---
import matplotlib.backends.backend_pgf as _pgf
if not hasattr(_pgf,"common_texification") and hasattr(_pgf,"_tex_escape"):
    _pgf.common_texification=_pgf._tex_escape
try:
    import webcolors as _wc
    if not hasattr(_wc,"CSS3_HEX_TO_NAMES"):
        try: _wc.CSS3_HEX_TO_NAMES={_wc.name_to_hex(n,spec="css3"):n for n in _wc.names("css3")}
        except Exception: _wc.CSS3_HEX_TO_NAMES={}
    if not hasattr(_wc,"CSS3_NAMES_TO_HEX"):
        _wc.CSS3_NAMES_TO_HEX={v:k for k,v in _wc.CSS3_HEX_TO_NAMES.items()}
except Exception: pass
try:
    import tikzplotlib
except Exception:
    tikzplotlib=None
import matplotlib.cm as cm; from matplotlib.colors import Normalize
OUT=f"cbag_out_{DATASET}"; SB="SBAR_results_AccessPoint"; FIG="FIG"; os.makedirs(FIG,exist_ok=True)
_PG=f"{OUT}/perclass_gate.csv"
GEN=sorted(pd.read_csv(_PG)["gen"].unique().tolist()) if os.path.exists(_PG) else ["C-SA","C-RL","C-PSO","C-GA"]
_PAL=["#2980b9","#27ae60","#c0392b","#8e44ad","#d68910","#16a085","#e67e22","#7f8c8d","#2c3e50","#c39bd3"]
GC={g:_PAL[i%len(_PAL)] for i,g in enumerate(GEN)}
MCOL=dict(DIFS="#8E44AD",CBAG="#D68910",SBAR="#16A085",Gate="#C0392B")
LOCK={'ip.proto','ip.ttl','radiotap.channel.freq'}
_pages=[]
def _save(f,name):
    f.savefig(f"{FIG}/{name}.pdf",bbox_inches="tight")
    if tikzplotlib is not None:
        try: tikzplotlib.save(f"{FIG}/{name}.tex",figure=f,axis_width="12cm")
        except Exception: pass
    _pages.append(f)
def short(s): return s.str.replace(r'.*\.','',regex=True)

### 3.2 — DIFS charts

In [ ]:
# ============================== DIFS ==============================
def difs_quadrant():
    d=pd.read_csv(f"{OUT}/DIFS_scores.csv"); f,ax=plt.subplots(figsize=(7,6))
    cols={'Core':'#c0392b','Vulnerable':'#e67e22','Ancillary':'#2980b9','Noise':'#95a5a6'}
    for q,c in cols.items():
        s=d[d.quadrant==q]; ax.scatter(s.MPI_norm,s.FPC_norm,c=c,s=55,label=q,edgecolor='k',lw=.3)
    ax.axvline(d.MPI_norm.median(),ls='--',c='gray'); ax.axhline(d.FPC_norm.median(),ls='--',c='gray')
    for _,r in d[d.quadrant=='Core'].sort_values('score',ascending=False).head(6).iterrows():
        ax.annotate(r.feature.split('.')[-1],(r.MPI_norm,r.FPC_norm),fontsize=7,xytext=(3,3),textcoords='offset points')
    ax.set_xlabel("MPI (Integrated-Gradients importance)"); ax.set_ylabel("FPC (permutation F1 drop)")
    ax.set_title("DIFS D1 - dual-impact quadrant (12 Core DIFs)"); ax.legend()
    _save(f,"D1_DIFS_quadrant")

def difs_feature_bars():
    d=pd.read_csv(f"{OUT}/DIFS_scores.csv").sort_values('score'); names=short(d.feature); norm=Normalize(0,1)
    f,ax=plt.subplots(figsize=(9,10)); ax.barh(names,d.score,color=cm.viridis(norm(d.MPI_norm)),edgecolor='k',lw=.3)
    sm=cm.ScalarMappable(norm=norm,cmap='viridis'); sm.set_array([]); f.colorbar(sm,ax=ax,label='MPI (importance)')
    ax.set_title("DIFS D2a - features (bar=dual score, colour=MPI)"); ax.tick_params(labelsize=7); _save(f,"D2a_features_colorMPI")
    f,ax=plt.subplots(figsize=(9,10)); ax.barh(names,d.score,color=cm.plasma(norm(d.FPC_norm)),edgecolor='k',lw=.3)
    sm=cm.ScalarMappable(norm=norm,cmap='plasma'); sm.set_array([]); f.colorbar(sm,ax=ax,label='FPC (F1 drop)')
    ax.set_title("DIFS D2b - features (bar=dual score, colour=FPC)"); ax.tick_params(labelsize=7); _save(f,"D2b_features_colorFPC")
    f,ax=plt.subplots(figsize=(9,10)); y=np.arange(len(d)); h=0.4
    ax.barh(y-h/2,d.MPI_norm,h,label='MPI',color='#2980b9'); ax.barh(y+h/2,d.FPC_norm,h,label='FPC',color='#c0392b')
    ax.set_yticks(y); ax.set_yticklabels(names,fontsize=7); ax.legend(); ax.set_title("DIFS D3 - MPI vs FPC per feature (grouped)"); _save(f,"D3_features_grouped")
    f,ax=plt.subplots(figsize=(9,10))
    ax.barh(names,d.MPI_norm,color='#2980b9',label='MPI ->'); ax.barh(names,-d.FPC_norm,color='#c0392b',label='<- FPC')
    ax.axvline(0,c='k',lw=.8); ax.set_xlim(-1.05,1.05); ax.legend(); ax.tick_params(labelsize=7)
    ax.set_title("DIFS D4 - diverging: FPC (left) | MPI (right)"); _save(f,"D4_features_diverging")

def difs_heatmap():
    d=pd.read_csv(f"{OUT}/DIFS_scores.csv").sort_values('score',ascending=False); core=d[d.quadrant=='Core']; M=core[['MPI_norm','FPC_norm','score']].values
    f,ax=plt.subplots(figsize=(6,7)); im=ax.imshow(M,cmap='YlOrRd',aspect='auto',vmin=0,vmax=1)
    ax.set_xticks(range(3)); ax.set_xticklabels(['MPI','FPC','score']); ax.set_yticks(range(len(core))); ax.set_yticklabels(short(core.feature),fontsize=8)
    for i in range(len(core)):
        for j in range(3): ax.text(j,i,f"{M[i,j]:.2f}",ha='center',va='center',fontsize=7)
    ax.set_title("DIFS D5 - 12 Core DIFs x {MPI, FPC, score}"); f.colorbar(im,ax=ax,fraction=0.046); _save(f,"D5_core_heatmap")

def difs_mutable():
    d=pd.read_csv(f"{OUT}/DIFS_scores.csv"); core=d[d.quadrant=='Core'].sort_values('score',ascending=False).copy()
    core['type']=core.feature.apply(lambda x:'locked' if x in LOCK else 'mutable')
    f,ax=plt.subplots(figsize=(8,5)); cc=['#c0392b' if t=='locked' else '#27ae60' for t in core.type]
    ax.barh(short(core.feature),core.score,color=cc,edgecolor='k'); ax.invert_yaxis()
    ax.set_title("DIFS D6 - Core DIFs: mutable (green,7) vs locked (red,3)"); ax.tick_params(labelsize=8); _save(f,"D6_mutable_locked")

### 3.3 — CBAG charts

In [ ]:
# ============================== CBAG (derived from perclass_gate.csv) ==============================
def _pg(): return pd.read_csv(f"{OUT}/perclass_gate.csv")

def cbag_funnel():
    d=_pg(); cand=int(d["n"].sum()); g1=int(d["G1_evade"].sum()); g2=int(d["G1G2_func"].sum()); fv=int(d["FVAS"].sum())
    stages=[("candidates",cand,MCOL['CBAG']),("Gate-1 evade",g1,'#e67e22'),("Gate-2 func",g2,'#e74c3c'),("Gate-3 FVAS",fv,MCOL['Gate'])]
    f,ax=plt.subplots(figsize=(9,5))
    for i,(lab,v,c) in enumerate(stages):
        ax.barh(3-i,v,color=c,edgecolor='k'); ax.text(v,3-i,f" {v:,}",va='center',fontsize=10)
    ax.set_yticks([3,2,1,0]); ax.set_yticklabels([s[0] for s in stages]); ax.set_title("CBAG C1 - tri-gate funnel"); _save(f,"C1_funnel")

def cbag_generators():
    d=_pg().groupby("gen").sum(numeric_only=True); G=[g for g in GEN if g in d.index]
    ev=100*d["G1_evade"]/d["n"]; fu=100*d["G1G2_func"]/d["n"]; fv=100*d["FVAS"]/d["n"]
    x=np.arange(len(G)); w=.26; f,ax=plt.subplots(figsize=(max(9,len(G)*1.1),5))
    ax.bar(x-w,ev[G],w,label='evasion%',color='#f39c12'); ax.bar(x,fu[G],w,label='functional%',color='#e74c3c'); ax.bar(x+w,fv[G],w,label='FVAS%',color='#c0392b')
    ax.set_xticks(x); ax.set_xticklabels(G,rotation=20); ax.legend(); ax.set_title("CBAG C2 - per-generator: evasion->functional->FVAS"); _save(f,"C2_generators")

def cbag_perclass():
    d=_pg(); pc=d.groupby('class')['FVAS'].sum(); pc=pc[pc>0].sort_values(ascending=False)
    if len(pc)==0: return
    f,ax=plt.subplots(figsize=(max(8,len(pc)*0.8),5)); ax.bar(pc.index,pc.values,color=MCOL['Gate'],edgecolor='k')
    for i,v in enumerate(pc.values): ax.text(i,v,str(int(v)),ha='center',va='bottom',fontsize=8)
    ax.set_title("CBAG C3 - FVAS survivors per class"); ax.tick_params(axis='x',rotation=25); _save(f,"C3_perclass_FVAS")

def cbag_fvr():
    d=_pg().groupby("gen").sum(numeric_only=True); G=[g for g in GEN if g in d.index]
    fvr=100*d["G1G2_func"]/d["G1_evade"].replace(0,np.nan)
    f,ax=plt.subplots(figsize=(max(8,len(G)*1.0),5)); ax.bar(range(len(G)),fvr[G].fillna(0),color='#16a085',edgecolor='k')
    ax.set_xticks(range(len(G))); ax.set_xticklabels(G,rotation=20); ax.set_ylabel('% functional among evaders')
    ax.set_title("CBAG C4 - functional validity rate"); _save(f,"C4_FVR")

def cbag_heatmap():
    d=_pg(); f,axs=plt.subplots(1,2,figsize=(max(10,len(GEN)*1.4),5))
    for ax,val,ttl,cmap in [(axs[0],"FVAS","(A) FVAS survivors","Reds"),(axs[1],"ASR_toNormal%","(B) evasion ASR %","YlOrRd")]:
        piv=d.pivot_table(index="class",columns="gen",values=val,aggfunc="sum" if val=="FVAS" else "mean").fillna(0)
        piv=piv.reindex(columns=[g for g in GEN if g in piv.columns]); piv=piv.reindex(index=piv.sum(1).sort_values(ascending=False).index)
        im=ax.imshow(piv.values,cmap=cmap,aspect="auto"); ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns,rotation=30,ha='right',fontsize=7)
        ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index,fontsize=7); ax.set_title(ttl); f.colorbar(im,ax=ax,fraction=0.046)
    f.suptitle("CBAG C5 - class x generator",fontweight='bold'); f.tight_layout(rect=[0,0,1,0.94]); _save(f,"C5_class_gen_heatmap")

def cbag_quadrant():
    p=f"{OUT}/DIFS_scores.csv"
    if not os.path.exists(p): return
    d=pd.read_csv(p); core=d[d.quadrant=='Core'].copy(); core['type']=core.feature.apply(lambda x:'locked' if x in LOCK else 'mutable')
    f,ax=plt.subplots(figsize=(7,6)); oth=d[d.quadrant!='Core']; ax.scatter(oth.MPI_norm,oth.FPC_norm,c='#bbb',s=20)
    for t,c,mk in [('mutable','#27ae60','o'),('locked','#c0392b','X')]:
        s=core[core.type==t]; ax.scatter(s.MPI_norm,s.FPC_norm,c=c,s=90,marker=mk,edgecolor='k',label=t)
    ax.axvline(d.MPI_norm.median(),ls='--',c='gray'); ax.axhline(d.FPC_norm.median(),ls='--',c='gray')
    ax.set_xlabel("MPI"); ax.set_ylabel("FPC"); ax.set_title("CBAG C6 - quadrant @ CBAG"); ax.legend(); _save(f,"C6_quadrant_CBAG")

def cbag_adaptive():
    p=f"{OUT}/gate3_L1_sweep.csv"
    if not os.path.exists(p): return
    sw=pd.read_csv(p); cols=[c for c in sw.columns if c.startswith('FVAS')]
    if not cols: return
    thr=[float(c.split('<')[-1]) for c in cols]; tot=[int(sw[c].sum()) for c in cols]
    f,ax=plt.subplots(figsize=(8,5)); ax.plot(thr,tot,'o-',color=MCOL['Gate'],lw=2)
    ax.set_xlabel("Gate-3 L1 budget"); ax.set_ylabel("FVAS"); ax.set_title("CBAG C7 - adaptive Gate-3"); ax.grid(alpha=.3); _save(f,"C7_adaptive_L1")

def cbag_stacked():
    d=_pg(); piv=d.pivot_table(index='class',columns='gen',values='FVAS',aggfunc='sum').fillna(0)
    piv=piv.reindex(columns=[g for g in GEN if g in piv.columns])
    if piv.shape[1]==0: return
    piv=piv[piv.sum(1)>0].sort_values(by=list(piv.columns),ascending=False)
    if len(piv)==0: return
    f,ax=plt.subplots(figsize=(max(9,len(piv)*0.8),5)); bottom=np.zeros(len(piv))
    for g in piv.columns:
        ax.bar(piv.index,piv[g],bottom=bottom,color=GC.get(g,'#999'),label=g,edgecolor='k',lw=.3); bottom+=piv[g].values
    ax.legend(title='generator',fontsize=7); ax.set_title("CBAG C8 - FVAS generator contribution per class"); ax.tick_params(axis='x',rotation=25); _save(f,"C8_stacked_generator")

def cbag_reachability():
    p=f"{OUT}/T3_reachability.csv"
    if not os.path.exists(p): return
    d=pd.read_csv(p); f,ax=plt.subplots(figsize=(8,5))
    ax.plot(d.L1_budget,d.FVAS_operating,'o-',lw=2,color=MCOL['CBAG'],label='operating gates (oracle)')
    ax.plot(d.L1_budget,d.FVAS_strict_CMR,'s--',lw=2,color=MCOL['Gate'],label='strict gates (oracle + CMR)')
    for x,y in zip(d.L1_budget,d.FVAS_operating): ax.annotate(f"{int(y)}",(x,y),fontsize=7,xytext=(0,6),textcoords='offset points',ha='center')
    ax.set_xlabel("Gate-3 L1 budget"); ax.set_ylabel("FVAS survivors")
    ax.set_title("CBAG C9 - reachability: survivors vs budget, operating vs strict gates")
    ax.legend(); ax.grid(alpha=.3); _save(f,"C9_reachability")

def cbag_tightening():
    p=f"{OUT}/T5_tightening.csv"; pa=f"{OUT}/A6_ranges.csv"
    if not os.path.exists(p): return
    d=pd.read_csv(p).sort_values('median_rho')
    f,axs=plt.subplots(1,2,figsize=(14,5))
    ax=axs[0]; ax.barh(d['class'],d['median_rho'],color=MCOL['CBAG'],edgecolor='k')
    ax.axvline(1.0,ls='--',c='k',lw=1); ax.set_xlabel("median rho = survivor width / allowed width")
    ax.set_title("(A) range tightening per class (rho<1 = concentrated)"); ax.tick_params(labelsize=8)
    ax=axs[1]
    if os.path.exists(pa):
        a=pd.read_csv(pa)
        ax.hist(a['rho'].dropna(),bins=30,color=MCOL['Gate'],edgecolor='k')
        ax.axvline(1.0,ls='--',c='k'); ax.set_xlabel("rho per (class, feature)")
        ax.set_title("(B) distribution of range tightening")
    else: ax.axis('off')
    f.suptitle("CBAG C10 - survivors occupy a much narrower region than the rulebook allows",fontweight='bold')
    f.tight_layout(rect=[0,0,1,0.93]); _save(f,"C10_tightening")

def cbag_vs_baseline():
    p=f"{OUT}/T7_constrained_vs_baseline.csv"
    if not os.path.exists(p): return
    d=pd.read_csv(p).sort_values(["type","FVAS%"],ascending=[True,False])
    col=['#c0392b' if t=="constrained" else '#7f8c8d' for t in d['type']]
    f,axs=plt.subplots(1,3,figsize=(16,5))
    for ax,c_,ttl in [(axs[0],"evasion%","(A) evasion % (baselines can match/beat)"),
                      (axs[1],"functional%_of_evaders","(B) FUNCTIONAL % of evaders  <- the CBAG claim"),
                      (axs[2],"FVAS%","(C) FVAS % (evade AND functional AND stealthy)")]:
        ax.bar(range(len(d)),d[c_],color=col,edgecolor='k')
        ax.set_xticks(range(len(d))); ax.set_xticklabels(d['gen'],rotation=40,ha='right',fontsize=7)
        ax.set_title(ttl,fontsize=10)
        for i,v in enumerate(d[c_]): ax.text(i,v,f"{v:.1f}",ha='center',va='bottom',fontsize=6)
    import matplotlib.patches as mp
    f.legend(handles=[mp.Patch(color='#c0392b',label='constrained (CBAG)'),
                      mp.Patch(color='#7f8c8d',label='unconstrained baseline')],
             loc='lower center',ncol=2,frameon=False)
    f.suptitle("CBAG C11 - constrained generation vs unconstrained baseline attacks",fontweight='bold')
    f.tight_layout(rect=[0,0.06,1,0.93]); _save(f,"C11_vs_baseline")

def cbag_perturbation_dist():
    p=f"{OUT}/T8_perturbation_dist.csv"
    if not os.path.exists(p): return
    d=pd.read_csv(p)
    f,axs=plt.subplots(1,2,figsize=(14,5))
    ax=axs[0]
    for i,r in d.iterrows():
        c='#c0392b' if r['type']=="constrained" else '#7f8c8d'
        ax.plot([r['L1_p10'],r['L1_p90']],[i,i],color=c,lw=2)
        ax.plot([r['L1_p25'],r['L1_p75']],[i,i],color=c,lw=6,alpha=.6)
        ax.plot(r['L1_p50'],i,'o',color='k',ms=5)
    ax.set_yticks(range(len(d))); ax.set_yticklabels(d['gen'],fontsize=8)
    ax.set_xlabel("L1 perturbation (p10-p90, box = p25-p75, dot = median)")
    ax.set_title("(A) perturbation size distribution")
    ax=axs[1]
    col=['#c0392b' if t=="constrained" else '#7f8c8d' for t in d['type']]
    ax.barh(d['gen'],d['L0_mean'],color=col,edgecolor='k')
    ax.set_xlabel("mean L0 (number of features changed)"); ax.set_title("(B) sparsity of the perturbation")
    ax.tick_params(labelsize=8)
    f.suptitle("CBAG C12 - how large and how sparse are the perturbations?",fontweight='bold')
    f.tight_layout(rect=[0,0,1,0.93]); _save(f,"C12_perturbation_dist")

def cbag_feature_freq():
    p=f"{OUT}/T9_feature_perturbation.csv"
    if not os.path.exists(p): return
    d=pd.read_csv(p)
    piv=d.pivot_table(index="feature",columns="class",values="frac_moved",aggfunc="mean").fillna(0)
    if piv.empty: return
    piv=piv.loc[piv.mean(1).sort_values(ascending=False).index]
    f,ax=plt.subplots(figsize=(max(7,piv.shape[1]*1.1),max(4,len(piv)*0.42)))
    im=ax.imshow(piv.values,cmap="Reds",aspect="auto",vmin=0,vmax=1)
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns,rotation=30,ha='right',fontsize=8)
    ax.set_yticks(range(len(piv))); ax.set_yticklabels([x.split('.')[-1] for x in piv.index],fontsize=8)
    for i in range(len(piv)):
        for j in range(piv.shape[1]): ax.text(j,i,f"{piv.values[i,j]:.2f}",ha='center',va='center',fontsize=6)
    ax.set_title("CBAG C13 - fraction of survivors in which each mutable DIF was moved")
    f.colorbar(im,ax=ax,fraction=0.046); _save(f,"C13_feature_perturbation")


### 3.4 — SBAR charts

In [ ]:
# ============================== SBAR ==============================
def _sbar_agg():
    rows=json.load(open(f"{SB}/holdout13_rows.json")); df=pd.DataFrame(rows)
    order=["Undefended","Naive-Aug","PGD-AT","Gentle-SBAR","No-HardNeg","Full-SBAR","No-Curriculum","EWC","Adaptive-Mix","Mix-15%","Mix-50%","No-Constraints","GFF-SBAR","Intersection-SBAR"]
    g=df.groupby('config').agg(group=('group','first'),clean=('clean','mean'),Normal=('Normal','mean'),residual=('residual','mean'),std=('residual','std'),wb=('whitebox','mean')).reindex(order)
    return g,order
def sbar_panel():
    g,order=_sbar_agg(); col=['#c0392b' if c=='Gentle-SBAR' else ('#95a5a6' if gr=='baseline' else ('#3498db' if gr=='ablation' else '#27ae60')) for c,gr in zip(g.index,g['group'])]
    f=plt.figure(figsize=(15,9)); f.suptitle("SBAR S1 - 13 configs, held-out C-PSO re-attack (3 seeds)",fontweight='bold')
    gs=f.add_gridspec(2,2,height_ratios=[1,1.3],hspace=0.5,wspace=0.22)
    ax=f.add_subplot(gs[0,0]); ax.bar(range(14),g['residual'],yerr=g['std'],color=col,edgecolor='k',capsize=3); ax.set_yscale('symlog',linthresh=0.1)
    ax.set_xticks(range(14)); ax.set_xticklabels(g.index,rotation=55,ha='right',fontsize=7.5); ax.set_title("residual ASR % (log)")
    ax=f.add_subplot(gs[0,1]); x=np.arange(14); w=.4; ax.bar(x-w/2,g['clean'],w,label='clean',color='#2980b9'); ax.bar(x+w/2,g['Normal'],w,label='Normal',color='#16a085')
    ax.set_xticks(x); ax.set_xticklabels(g.index,rotation=55,ha='right',fontsize=7.5); ax.set_ylim(55,101); ax.legend(fontsize=8); ax.set_title("benign accuracy")
    ax=f.add_subplot(gs[1,:]); ax.axis('off'); cols=['config','group','clean%','Normal%','residual%','white-box%']
    cells=[[c,g.loc[c,'group'],f"{g.loc[c,'clean']:.1f}",f"{g.loc[c,'Normal']:.1f}",f"{g.loc[c,'residual']:.2f}±{g.loc[c,'std']:.2f}",f"{g.loc[c,'wb']:.1f}"] for c in order]
    tb=ax.table(cellText=cells,colLabels=cols,loc='center',cellLoc='center'); tb.auto_set_font_size(False); tb.set_fontsize(8); tb.scale(1,1.35)
    for j in range(len(cols)): tb[0,j].set_facecolor('#34495e'); tb[0,j].set_text_props(color='white',weight='bold')
    gr=order.index('Gentle-SBAR')+1
    for j in range(len(cols)): tb[gr,j].set_facecolor('#f9d7d2'); tb[gr,j].set_text_props(weight='bold')
    _save(f,"S1_SBAR_panel")
def sbar_h3():
    if not os.path.exists(f"{SB}/H3_heldout.csv"): return
    H=pd.read_csv(f"{SB}/H3_heldout.csv"); f,ax=plt.subplots(figsize=(9,5)); x=np.arange(len(H))
    ax.bar(x-0.2,H['ASR_undef%'],0.4,label='Undefended',color='#95a5a6'); ax.bar(x+0.2,H['ASR_Gentle%'],0.4,label='Gentle-SBAR',color='#16a085')
    ax.set_xticks(x); ax.set_xticklabels(H['class'],rotation=25,ha='right'); ax.legend(); ax.set_title("SBAR S2 - H3 per-class held-out re-attack"); _save(f,"S2_H3")
def sbar_ranking():
    g,order=_sbar_agg(); und=g.loc['Undefended','residual']; g['red']=und/g['residual'].replace(0,np.nan)
    gg=g.dropna(subset=['red']).sort_values('red'); f,ax=plt.subplots(figsize=(9,5)); cc=['#c0392b' if c=='Gentle-SBAR' else '#3498db' for c in gg.index]
    ax.barh(gg.index,gg['red'],color=cc,edgecolor='k'); ax.set_xlabel("reduction ratio (x vs undefended)"); ax.set_title("SBAR S3 - reduction ranking (non-collapsed defenses)")
    for i,v in enumerate(gg['red']): ax.text(v,i,f" {v:.0f}x",va='center',fontsize=8)
    _save(f,"S3_reduction_ranking")
def sbar_weightcurve():
    M=np.linspace(0,0.1,100); wf=1/(0.1-M+0.001); wg=1+4*(M/0.1)
    f,ax=plt.subplots(figsize=(8,5)); ax.plot(M,wf,label='Full-SBAR (unbounded)',color='#c0392b',lw=2)
    ax.plot(M,wg,label='Gentle-SBAR (bounded [1,5])',color='#16a085',lw=2); ax.plot(M,np.ones_like(M),label='No-HardNeg (w=1)',color='#95a5a6',lw=2,ls='--')
    ax.set_yscale('log'); ax.set_xlabel("margin M = min(|thr - P(N)|, 0.1)"); ax.set_ylabel("training weight w (log)")
    ax.set_title("SBAR S4 - hard-negative weight vs margin"); ax.legend(); _save(f,"S4_weight_curve")

# ---- performance-metric demonstration figures (reporting only; derived from holdout13_rows.json) ----
def _sbar_df():
    return pd.DataFrame(json.load(open(f"{SB}/holdout13_rows.json")))

def sbar_tradeoff():
    df=_sbar_df()
    if 'macro_f1' not in df.columns: return   # needs a run with metrics wired in
    g=df.groupby('config').agg(mf1=('macro_f1','mean'),res=('residual','mean'),grp=('group','first'))
    f,ax=plt.subplots(figsize=(8,6))
    for cfg,r in g.iterrows():
        col='#c0392b' if cfg=='Gentle-SBAR' else ('#95a5a6' if r['grp']=='baseline' else ('#3498db' if r['grp']=='ablation' else '#27ae60'))
        ax.scatter(r['mf1'],r['res'],s=90,color=col,edgecolor='k',zorder=3)
        ax.annotate(cfg,(r['mf1'],r['res']),fontsize=7,xytext=(4,3),textcoords='offset points')
    ax.set_yscale('symlog',linthresh=0.1); ax.set_xlabel('clean macro-F1 % (higher better) ->'); ax.set_ylabel('residual ASR % (lower better)')
    ax.invert_yaxis(); ax.set_title("SBAR S5 - clean macro-F1 vs residual (top-right = best trade-off)"); ax.grid(alpha=.3)
    _save(f,"S5_tradeoff")

def sbar_class_config_heatmap():
    df=_sbar_df()
    if 'per_class' not in df.columns: return
    rows={}
    for _,r in df.iterrows():
        pc=r['per_class'] if isinstance(r['per_class'],dict) else {}
        rows.setdefault(r['config'],{})
        for k,v in pc.items(): rows[r['config']].setdefault(k,[]).append(v)
    order=["Undefended","Naive-Aug","PGD-AT","Gentle-SBAR","No-HardNeg","Full-SBAR","No-Curriculum","EWC","Adaptive-Mix","Mix-15%","Mix-50%","No-Constraints","GFF-SBAR","Intersection-SBAR"]
    cfgs=[c for c in order if c in rows]; cls=sorted({k for c in rows for k in rows[c]})
    if not cfgs or not cls: return
    M=np.array([[np.mean(rows[c].get(k,[0])) for k in cls] for c in cfgs])
    f,ax=plt.subplots(figsize=(max(7,len(cls)*1.1),max(5,len(cfgs)*0.5)))
    im=ax.imshow(M,cmap='YlOrRd',aspect='auto')
    ax.set_xticks(range(len(cls))); ax.set_xticklabels(cls,rotation=30,ha='right',fontsize=7); ax.set_yticks(range(len(cfgs))); ax.set_yticklabels(cfgs,fontsize=7)
    for i in range(len(cfgs)):
        for j in range(len(cls)): ax.text(j,i,f"{M[i,j]:.1f}",ha='center',va='center',fontsize=6)
    ax.set_title("SBAR S6 - residual ASR % : config x attack class"); f.colorbar(im,ax=ax,fraction=0.046)
    _save(f,"S6_class_config_heatmap")

def sbar_metrics_bars():
    df=_sbar_df()
    have=[m for m in ['macro_f1','bal_acc','DR','FPR'] if m in df.columns]
    if not have: return
    order=["Undefended","Naive-Aug","PGD-AT","Gentle-SBAR","No-HardNeg","Full-SBAR","No-Constraints"]
    g=df.groupby('config').mean(numeric_only=True); cfgs=[c for c in order if c in g.index]
    if not cfgs: return
    f,ax=plt.subplots(figsize=(max(9,len(cfgs)*1.3),5)); x=np.arange(len(cfgs)); w=0.8/len(have)
    cols={'macro_f1':'#16a085','bal_acc':'#2980b9','DR':'#8e44ad','FPR':'#c0392b'}
    for k,m in enumerate(have):
        ax.bar(x+(k-len(have)/2)*w+w/2,g.loc[cfgs,m],w,label=m,color=cols.get(m,'#777'))
    ax.set_xticks(x); ax.set_xticklabels(cfgs,rotation=25,ha='right',fontsize=8); ax.legend(fontsize=8)
    ax.set_title("SBAR S7 - IDS metrics per config (macro-F1 / balanced-acc / DR / FPR)"); _save(f,"S7_metrics_bars")

def target_metrics_fig():
    p=f"{OUT}/target_perclass_metrics.csv"
    if not os.path.exists(p): return
    d=pd.read_csv(p).sort_values('recall'); f,ax=plt.subplots(figsize=(8,max(4,len(d)*0.4)))
    y=np.arange(len(d)); h=0.4
    ax.barh(y-h/2,d['recall'],h,label='recall (detection)',color='#16a085'); ax.barh(y+h/2,d['precision'],h,label='precision',color='#2980b9')
    ax.set_yticks(y); ax.set_yticklabels(d['class'],fontsize=8); ax.set_xlim(0,1.02); ax.legend(); ax.set_title("Target IDS - per-class precision & recall (detection)")
    _save(f,"T0_target_perclass")

def results_tex():
    p=f"{SB}/E1_heldout_13configs.csv"
    if not os.path.exists(p): return
    d=pd.read_csv(p)
    try:
        d.round(3).to_latex(f"{SB}/results_sbar.tex",index=False,
            caption="SBAR under held-out C-PSO re-attack (mean over 3 seeds).",label="tab:sbar-heldout")
    except Exception: d.round(3).to_csv(f"{SB}/results_sbar_fallback.csv",index=False)


### 3.5 — table pages + run_all_figures()

In [ ]:
# ============================== TABLE PAGES ==============================
def _table_page(title,df,name,fontsize=7):
    df=df.copy()
    for c in df.columns:
        try: df[c]=df[c].map(lambda v:f"{v:.3f}" if isinstance(v,float) else v)
        except Exception: pass
    nrow,ncol=df.shape; f,ax=plt.subplots(figsize=(min(2+1.4*ncol,16),min(1.2+0.32*nrow,20))); ax.axis('off')
    ax.set_title(title,fontweight='bold',fontsize=11,pad=12)
    tb=ax.table(cellText=df.values,colLabels=list(df.columns),loc='center',cellLoc='center')
    tb.auto_set_font_size(False); tb.set_fontsize(fontsize); tb.scale(1,1.3)
    for j in range(ncol): tb[0,j].set_facecolor('#34495e'); tb[0,j].set_text_props(color='white',weight='bold')
    _save(f,name)

def result_tables():
    d=pd.read_csv(f"{OUT}/DIFS_scores.csv")
    core=d[d.quadrant=='Core'].sort_values('score',ascending=False)[['feature','MPI_norm','FPC_norm','score']]
    _table_page("TABLE T1 - DIFS: 12 Core DIFs (MPI, FPC, dual score)",core,"T1_DIFS_core")
    evf=pd.read_csv(f"{OUT}/evasion_vs_functionality.csv"); fvr=pd.read_csv(f"{OUT}/cbag_FVR.csv"); qe=pd.read_csv(f"{OUT}/query_efficiency.csv")
    g=evf.merge(fvr[['gen','FVR_oracle%','FVR_CMR%','L1_mean']],on='gen').merge(qe[['gen','FVAS_per_1k_queries']],on='gen')
    _table_page("TABLE T2 - CBAG: per-generator (evasion/functional/FVAS/FVR/L1/queries)",g,"T2_CBAG_generators")
    gate=pd.read_csv(f"{OUT}/perclass_gate.csv"); _table_page("TABLE T3 - CBAG: per class x generator gate flow",gate,"T3_CBAG_perclass",fontsize=6)
    sw=pd.read_csv(f"{OUT}/gate3_L1_sweep.csv"); cols=[c for c in sw.columns if c.startswith('FVAS')]
    tot=pd.DataFrame({'L1 budget':[c.split('<')[-1] for c in cols],'total FVAS':[int(sw[c].sum()) for c in cols]})
    _table_page("TABLE T4 - CBAG: adaptive Gate-3 L1 sweep (totals)",tot,"T4_CBAG_L1sweep")
    if os.path.exists(f"{SB}/E1_heldout_13configs.csv"):
        E=pd.read_csv(f"{SB}/E1_heldout_13configs.csv"); _table_page("TABLE T5 - SBAR: 13 configs (held-out C-PSO, 3 seeds)",E,"T5_SBAR_E1",fontsize=7)
    if os.path.exists(f"{SB}/H3_heldout.csv"):
        H=pd.read_csv(f"{SB}/H3_heldout.csv"); _table_page("TABLE T6 - SBAR: H3 per-class (Undefended -> Gentle-SBAR)",H,"T6_SBAR_H3")

def run_all_figures():
    global _pages; _pages=[]
    _fns=[difs_quadrant,difs_feature_bars,difs_heatmap,difs_mutable,
          cbag_funnel,cbag_generators,cbag_perclass,cbag_fvr,cbag_heatmap,cbag_quadrant,cbag_adaptive,cbag_stacked,cbag_reachability,cbag_tightening,cbag_vs_baseline,cbag_perturbation_dist,cbag_feature_freq,
          sbar_panel,sbar_h3,sbar_ranking,sbar_weightcurve,
          sbar_tradeoff,sbar_class_config_heatmap,sbar_metrics_bars,target_metrics_fig,results_tex,result_tables]
    for fn in _fns:
        try: fn()
        except FileNotFoundError as e: print(f"[skip {fn.__name__}] missing file: {e.filename}")
        except Exception as e: print(f"[skip {fn.__name__}] {type(e).__name__}: {str(e)[:90]}")
    with PdfPages(f"{FIG}/ALL_RESULTS.pdf") as pp:
        for fg in _pages: pp.savefig(fg)
    plt.close('all'); print(f"[figures] {len(_pages)} pages -> {FIG}/  +  {FIG}/ALL_RESULTS.pdf")

### ▶ Run SECTION 3

In [ ]:
run_all_figures()
print('SECTION 3 done — PDFs + TikZ in ./FIG + ALL_RESULTS.pdf')

# SECTION 4 — Package results as one zip

In [ ]:
# ---- package results into one zip (PDFs + TikZ; no PNGs) ----
import os, zipfile, datetime
PACK = "auto"     # "cbag" | "sbar" | "both" | "auto"  (auto = whatever exists)
STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M")
CODE  = ["AccessPoint_full_pipeline.ipynb", "run_cbag.py", "run_sbar_holdout13.py", "figures_all.py"]
_have_cbag = os.path.isdir("cbag_out"); _have_sbar = os.path.isdir("SBAR_results_AccessPoint")
mode = PACK if PACK!="auto" else ("both" if (_have_cbag and _have_sbar) else ("cbag" if _have_cbag else "sbar"))
SRC = {"cbag": ["cbag_out"], "sbar": ["SBAR_results_AccessPoint"], "both": ["cbag_out","SBAR_results_AccessPoint"]}[mode] + ["FIG"] + CODE
ZIP = f"AccessPoint_{mode}_results_{STAMP}.zip"
SKIP_EXT = (".pkl", ".png")      # skip big caches + PNGs; keep PDFs + TikZ .tex
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for s in SRC:
        if os.path.isfile(s): z.write(s)
        elif os.path.isdir(s):
            for root,_,files in os.walk(s):
                for fn in files:
                    if fn.endswith(SKIP_EXT): continue
                    z.write(os.path.join(root, fn))
print(f"packaged [{mode}] ->", ZIP, f"({os.path.getsize(ZIP)/1e6:.1f} MB,", len(zipfile.ZipFile(ZIP).namelist()), "files)")